# 장중(60분·10분·5분봉) 상승·하락 변동성과 이후 변동성 — 검정 보고서

- 작성일: 2026-09-28
- 코드: `scripts/step7b_intraday_semivar.py` (공통 함수 `src/semivar.py`, 장중 지표 `src/vol.py`)
- 원자료: `results/step7b/*.csv`
- 관련 보고서: 일봉 기반 분석 `REPORT_semivariance.md`

---

## 0. 요약

질문: **오늘 장중(5분·10분·60분봉)의 상승 변동성과 하락 변동성이 내일 이후의 상승·하락 변동성과 어떤 관계이고, 그 영향이 유의한가.**

| 분석 | 결과 |
|---|---|
| **60분봉 3년, 미국 지수** | 과거 하락 쪽 총영향은 양(+)이고(미래 RV 1일 2.70, p = 0.072 / 5일 1.50, p = 0.059), 상승 쪽은 0 근처다. 방향은 일봉 결과와 같지만 **하락 − 상승 차이는 유의하지 않다** (p = 0.15 ~ 0.18) |
| **60분봉 3년, 미국 50종목** | **하락 총영향은 9개 조합 모두 유의**(p ≤ 0.030)하고, 상승 총영향은 1개만 유의하다. 차이는 양(+)이지만 유의하지 않다 (p ≥ 0.10) |
| **60분봉 3년, 한국 지수** | **반대 방향.** 장중 **상승** 변동성이 이후 하락·전체 변동성을 키운다 (미래 RS⁻ 1일 Σβ⁺ = 3.19, p < 0.001). 하락 − 상승 차이는 음(−)으로 유의하다 (p = 0.002 ~ 0.031) |
| **60분봉 3년, 한국 50종목** | **대칭.** 상승·하락 총영향이 모두 유의하고(p < 0.02) 크기가 거의 같다 (차이 p ≥ 0.41) |
| **밤사이 분리 (60분봉)** | 미국 지수에서 하락 우위는 **밤사이 하락**(전일 종가 → 당일 시가)에서 유의하다 (Σβ⁻ − Σβ⁺: 미래 RV 1·5·22일 p = 0.031 / 0.021 / 0.032). **장중** 차이는 유의하지 않다 |
| **같은 기간 일봉 대조군** | 같은 날짜를 종가 대 종가(밤사이 포함)로 재면 미국은 하락 우위가 유의하다 (미래 RV 1일 p = 0.017, 5일 p = 0.039). **기간이 아니라 측정 방식(장중만 보느냐)이 결과를 가른다** |
| **5분·10분봉 60일** | 결과가 봉 간격(5분/10분)과 예측 지평(1/5일)에 따라 뒤집혀 **믿을 만한 결론이 없다.** 미국 지수는 상승 우위(p ≈ 0.01)이고, 같은 60일의 일봉 대조군은 하락 우위(p ≈ 0.006)다. 예비 결과로만 본다 |

**정리하면**
1. 일봉 분석에서 확인한 "하락 변동성이 이후 변동성을 더 키운다"는 비대칭은, 미국에서는 상당 부분 **밤사이 하락(갭 하락)** 에서 나온다.
2. 장이 열려 있는 동안의 상승·하락 움직임은 이후 변동성을 **비슷하게** 키운다. 미국 종목과 한국 종목 모두 두 방향이 다 유의하고, 차이는 유의하지 않다.
3. 한국 지수는 2023~2026년이 데이터상 **극단적 변동 국면**(3년 +182%, 연변동성 35%, 하루 ±8% 초과 11회)이다. 이 기간에는 장중 상승 변동성이 이후 변동성을 오히려 더 크게 예고했다.
4. 5분·10분봉 60일 분석은 표본이 너무 짧아 결론을 내릴 수 없다.

---

## 1. 데이터와 방법

### 1.1 장중 상승·하락 반분산

하루 안의 연속된 봉 종가 사이 로그수익률을 r₁, r₂, …, r_N(%)이라고 할 때:

- **장중 상승 반분산** RS⁺ = Σ rᵢ² (rᵢ > 0인 구간만)
- **장중 하락 반분산** RS⁻ = Σ rᵢ² (rᵢ < 0인 구간만)
- RS⁺ + RS⁻ = 그날의 장중 실현분산 RV

**전일 마지막 봉 → 당일 첫 봉 수익률(밤사이, ON)은 장중 지표에서 제외된다.** 3장에서는 이것을 따로 ON⁺ = ON² (ON > 0), ON⁻ = ON² (ON < 0)로 만들어 함께 넣는다.

| 봉 | 원천 | 하루 수익률 개수 | 기간 |
|---|---|---|---|
| 60분봉 | yfinance 60분봉 | 미국 6, 한국 5 | 약 3년 (2023-09/10 ~ 2026-09) |
| 10분봉 | 5분봉 종가를 10분 구간의 마지막 값으로 묶음 | 미국 38, 한국 35 | 약 60거래일 (2026-06/07 ~ 2026-09) |
| 5분봉 | yfinance 5분봉 | 미국 77, 한국 71 | 약 60거래일 |

반일장이나 결측이 많은 날(그 종목 하루 봉 수 중앙값의 80% 미만)은 제외했다. yfinance는 10분봉을 제공하지 않는다.

### 1.2 회귀식 (일봉 분석과 같은 구조)

```
미래 y(t+1 ~ t+h) = c + Σ_k [ β⁺_k RS⁺_k,t + β⁻_k RS⁻_k,t ] + ε      (k = 일, 주, 월)
y ∈ {미래 장중 RS⁺, 미래 장중 RS⁻, 미래 장중 RV}
```

| 설정 (미리 정함) | 60분봉 | 5분·10분봉 |
|---|---|---|
| 설명변수 | 일, 주(5일 평균), 월(22일 평균) | 일, 주(5일 평균) — 60일로는 월 평균을 만들 여유가 없음 |
| 예측 지평 h | 1, 5, 22일 | 1, 5일 |
| 표준오차 시차 | max(22, 2h) | max(5, 2h) |

### 1.3 분석 단위

| 단위 | 내용 | 표준오차 |
|---|---|---|
| **지수** | S&P 500(`^GSPC`), KOSPI(`^KS11`) 단독 시계열. 단위 %² | Newey-West |
| ETF | SPY, KODEX 200(`069500.KS`). 지수 결과 확인용 | Newey-West |
| **종목 패널** | 시장별 대형주 50개를 묶은 합동 회귀. 종목마다 변동성 수준이 달라서, 각 종목의 RS±를 그 종목의 평균 일간 RV로 나눠 단위를 없앴다 | **Driscoll-Kraay** (같은 날 종목 간 상관과 날짜 간 자기상관에 모두 강건) |
| **일봉 대조군** | 같은 기간·같은 날짜의 지수 일봉(종가 대 종가, 밤사이 포함) 반분산으로 같은 회귀 | Newey-West |

일봉 대조군은 "결과 차이가 **기간** 때문인가, **측정 방식** 때문인가"를 가르기 위한 것이다.

### 1.4 검정

일봉 보고서와 같다.
- 개별 계수 β = 0
- 총영향 Σβ⁺ = 0, Σβ⁻ = 0
- 방향 차이 Σβ⁻ − Σβ⁺ = 0 (양수면 하락의 영향이 더 큼)
- 항목별·동시 β⁻ = β⁺ (Wald χ²)

모든 p값은 양측이다. 표기는 `*` p < 0.05, `**` p < 0.01, `***` p < 0.001이다.

### 1.5 표본

| 봉 | 시장 | 단위 | 종목 수 | 거래일 | 기간 | 하루 봉 수(중앙값) |
|---|---|---|---:|---:|---|---:|
| 60m | US | 지수 | 1 | 721 | 2023-10-27 ~ 2026-09-25 | 7 |
| 60m | US | ETF | 1 | 721 | 2023-10-27 ~ 2026-09-25 | 7 |
| 60m | US | 지수 일봉(대조군) | 1 | 730 | 2023-10-27 ~ 2026-09-25 |  |
| 60m | US | 종목 패널 | 50 | 721 | 2023-10-27 ~ 2026-09-25 |  |
| 60m | KR | 지수 | 1 | 728 | 2023-09-21 ~ 2026-09-23 | 6 |
| 60m | KR | ETF | 1 | 728 | 2023-09-21 ~ 2026-09-23 | 6 |
| 60m | KR | 지수 일봉(대조군) | 1 | 730 | 2023-09-21 ~ 2026-09-23 |  |
| 60m | KR | 종목 패널 | 50 | 728 | 2023-09-21 ~ 2026-09-23 |  |
| 5m | US | 지수 | 1 | 60 | 2026-07-02 ~ 2026-09-25 | 78 |
| 5m | US | ETF | 1 | 60 | 2026-07-02 ~ 2026-09-25 | 78 |
| 5m | US | 지수 일봉(대조군) | 1 | 60 | 2026-07-02 ~ 2026-09-25 |  |
| 5m | US | 종목 패널 | 50 | 60 | 2026-07-02 ~ 2026-09-25 |  |
| 5m | KR | 지수 | 1 | 59 | 2026-06-30 ~ 2026-09-23 | 72 |
| 5m | KR | ETF | 1 | 60 | 2026-06-30 ~ 2026-09-23 | 72 |
| 5m | KR | 지수 일봉(대조군) | 1 | 60 | 2026-06-30 ~ 2026-09-23 |  |
| 5m | KR | 종목 패널 | 50 | 60 | 2026-06-30 ~ 2026-09-23 |  |
| 10m | US | 지수 | 1 | 60 | 2026-07-02 ~ 2026-09-25 | 39 |
| 10m | US | ETF | 1 | 60 | 2026-07-02 ~ 2026-09-25 | 39 |
| 10m | US | 지수 일봉(대조군) | 1 | 60 | 2026-07-02 ~ 2026-09-25 |  |
| 10m | US | 종목 패널 | 50 | 60 | 2026-07-02 ~ 2026-09-25 |  |
| 10m | KR | 지수 | 1 | 59 | 2026-06-30 ~ 2026-09-23 | 36 |
| 10m | KR | ETF | 1 | 60 | 2026-06-30 ~ 2026-09-23 | 36 |
| 10m | KR | 지수 일봉(대조군) | 1 | 60 | 2026-06-30 ~ 2026-09-23 |  |
| 10m | KR | 종목 패널 | 50 | 60 | 2026-06-30 ~ 2026-09-23 |  |

### 1.6 표본 기간의 시장 상황 (지수 일봉 기준)

| 기간 | 시장 | 지수 수익률 | 연변동성 | ±3% 초과일 (상승 / 하락) | 최대 / 최소 일수익률 |
|---|---|---:|---:|---|---|
| 60분봉 3년 | 미국 | +88.1% | 15.0% | 2 / 4 | +9.09% / −6.16% |
| 60분봉 3년 | 한국 | **+181.6%** | **35.4%** | **35 / 42** | +16.48% / −12.85% |
| 5·10분봉 60일 | 미국 | +3.3% | 11.1% | 0 / 0 | +1.77% / −1.53% |
| 5·10분봉 60일 | 한국 | −16.5% | **69.1%** | 10 / 15 | +16.48% / −11.47% |

데이터상 KOSPI는 2023-09 약 2,400에서 2026-06 약 9,100까지 올랐다가, 2026-07 말 약 5,600까지 급락했다. 2026-03 ~ 07에만 하루 ±8%를 넘는 날이 10번 있다 (3년 전체 11번). 일봉 품질 점검에서는 오류가 발견되지 않았지만, 실제 시장 기록과 대조하지는 않았다. **한국 결과는 이런 극단적 국면에서 나온 것**이라는 점을 전제로 읽어야 한다. 미국 60일 구간은 반대로 매우 조용한 시기다(±3% 초과일 0).

---

## 2. 60분봉 3년 결과

각 칸은 추정값 (t, p)다.

### 2.1 미국 지수 (S&P 500)

| 목표 | h | 상승 총영향 Σβ⁺ | 하락 총영향 Σβ⁻ | 차이 Σβ⁻ − Σβ⁺ | β⁻ = β⁺ 동시 χ² (p) |
|---|---:|---:|---:|---:|---:|
| 미래 RS⁺ | 1 | -0.702 (t -1.04, p 0.297) | 2.502 (t 1.79, p 0.073) | 3.204 (t 1.55, p 0.121) | 9.5* (0.023) |
| 미래 RS⁺ | 5 | -0.252 (t -0.93, p 0.351) | 1.156* (t 2.07, p 0.038) | 1.408 (t 1.72, p 0.086) | 11.0* (0.012) |
| 미래 RS⁺ | 22 | -0.790 (t -1.28, p 0.201) | 1.787 (t 1.50, p 0.133) | 2.577 (t 1.43, p 0.153) | 10.7* (0.014) |
| 미래 RS⁻ | 1 | 0.263** (t 2.69, p 0.007) | 0.193 (t 1.13, p 0.260) | -0.069 (t -0.26, p 0.794) | 28.8*** (<0.001) |
| 미래 RS⁻ | 5 | 0.068 (t 0.53, p 0.596) | 0.342 (t 1.33, p 0.185) | 0.274 (t 0.72, p 0.471) | 6.6 (0.084) |
| 미래 RS⁻ | 22 | -0.306 (t -0.93, p 0.354) | 0.800 (t 1.26, p 0.208) | 1.106 (t 1.15, p 0.252) | 6.0 (0.111) |
| 미래 RV | 1 | -0.439 (t -0.60, p 0.547) | 2.695 (t 1.80, p 0.072) | 3.134 (t 1.41, p 0.158) | 7.1 (0.068) |
| 미래 RV | 5 | -0.185 (t -0.48, p 0.631) | 1.498 (t 1.89, p 0.059) | 1.682 (t 1.44, p 0.149) | 12.5** (0.006) |
| 미래 RV | 22 | -1.096 (t -1.16, p 0.247) | 2.587 (t 1.42, p 0.155) | 3.683 (t 1.33, p 0.183) | 9.5* (0.024) |

- 하락 쪽 총영향은 모두 양(+)이다. 미래 RS⁺ 5일에서는 유의하다 (1.156, p = 0.038).
- 상승 쪽은 대부분 음(−)이거나 0 근처다. 미래 RS⁻ 1일만 0.263 (p = 0.007)이다.
- 하락 − 상승 차이는 모두 양(+)(1.4 ~ 3.7)이지만 **유의하지 않다** (p = 0.09 ~ 0.25).
- ETF(SPY)도 거의 같은 수치다 (부록 B).

### 2.2 한국 지수 (KOSPI)

| 목표 | h | 상승 총영향 Σβ⁺ | 하락 총영향 Σβ⁻ | 차이 Σβ⁻ − Σβ⁺ | β⁻ = β⁺ 동시 χ² (p) |
|---|---:|---:|---:|---:|---:|
| 미래 RS⁺ | 1 | 0.189 (t 0.28, p 0.779) | 0.529 (t 0.95, p 0.341) | 0.340 (t 0.28, p 0.781) | 3.2 (0.356) |
| 미래 RS⁺ | 5 | 0.351 (t 0.46, p 0.649) | 0.347 (t 0.53, p 0.598) | -0.004 (t -0.00, p 0.998) | 10.3* (0.016) |
| 미래 RS⁺ | 22 | 0.648* (t 2.19, p 0.028) | 0.055 (t 0.18, p 0.854) | -0.594 (t -1.03, p 0.303) | 1.5 (0.693) |
| 미래 RS⁻ | 1 | 3.185*** (t 3.70, p <0.001) | -1.502* (t -2.31, p 0.021) | -4.686** (t -3.12, p 0.002) | 12.0** (0.007) |
| 미래 RS⁻ | 5 | 2.760*** (t 3.88, p <0.001) | -1.223* (t -2.11, p 0.035) | -3.983** (t -3.11, p 0.002) | 15.4** (0.001) |
| 미래 RS⁻ | 22 | 1.051* (t 2.54, p 0.011) | -0.038 (t -0.08, p 0.933) | -1.088 (t -1.29, p 0.197) | 8.8* (0.031) |
| 미래 RV | 1 | 3.373** (t 2.96, p 0.003) | -0.973 (t -1.09, p 0.274) | -4.346* (t -2.15, p 0.031) | 7.9* (0.048) |
| 미래 RV | 5 | 3.111* (t 2.44, p 0.015) | -0.876 (t -0.81, p 0.420) | -3.986 (t -1.70, p 0.090) | 10.8* (0.013) |
| 미래 RV | 22 | 1.699* (t 2.49, p 0.013) | 0.017 (t 0.02, p 0.981) | -1.682 (t -1.22, p 0.223) | 3.6 (0.308) |

- **일봉 결과와 반대다.** 장중 상승 변동성의 총영향이 미래 하락 변동성(1일 3.185, 5일 2.760, 22일 1.051, p ≤ 0.011)과 미래 전체 변동성(p ≤ 0.015)에서 양(+)으로 유의하다.
- 장중 하락 변동성은 미래 하락 변동성에 음(−)이다 (1일 −1.502, p = 0.021).
- 그래서 차이 Σβ⁻ − Σβ⁺가 음(−)으로 유의하다 (미래 RS⁻ 1일 −4.69, p = 0.002 / 5일 −3.98, p = 0.002).
- KODEX 200 ETF도 같은 패턴이다 (부록 B).

### 2.3 종목 패널 (시장별 50종목, 종목 평균 RV로 정규화)

#### 미국

| 목표 | h | 상승 총영향 Σβ⁺ | 하락 총영향 Σβ⁻ | 차이 Σβ⁻ − Σβ⁺ | β⁻ = β⁺ 동시 χ² (p) |
|---|---:|---:|---:|---:|---:|
| 미래 RS⁺ | 1 | 0.064 (t 0.45, p 0.651) | 0.705* (t 2.17, p 0.030) | 0.641 (t 1.40, p 0.163) | 3.2 (0.368) |
| 미래 RS⁺ | 5 | 0.066 (t 0.59, p 0.559) | 0.468** (t 2.70, p 0.007) | 0.402 (t 1.50, p 0.133) | 2.6 (0.454) |
| 미래 RS⁺ | 22 | -0.061 (t -0.40, p 0.691) | 0.455* (t 2.25, p 0.024) | 0.516 (t 1.53, p 0.126) | 2.6 (0.462) |
| 미래 RS⁻ | 1 | 0.208*** (t 3.34, p <0.001) | 0.309*** (t 4.15, p <0.001) | 0.101 (t 0.82, p 0.410) | 30.1*** (<0.001) |
| 미래 RS⁻ | 5 | 0.097 (t 1.16, p 0.244) | 0.318*** (t 3.50, p <0.001) | 0.220 (t 1.40, p 0.161) | 3.4 (0.337) |
| 미래 RS⁻ | 22 | -0.016 (t -0.14, p 0.887) | 0.322** (t 2.90, p 0.004) | 0.338 (t 1.65, p 0.098) | 3.5 (0.318) |
| 미래 RV | 1 | 0.271 (t 1.40, p 0.163) | 1.014** (t 2.61, p 0.009) | 0.742 (t 1.31, p 0.191) | 5.5 (0.140) |
| 미래 RV | 5 | 0.163 (t 0.84, p 0.399) | 0.785** (t 3.02, p 0.002) | 0.622 (t 1.49, p 0.137) | 2.8 (0.425) |
| 미래 RV | 22 | -0.077 (t -0.29, p 0.771) | 0.778* (t 2.50, p 0.012) | 0.855 (t 1.58, p 0.113) | 2.7 (0.438) |

#### 한국

| 목표 | h | 상승 총영향 Σβ⁺ | 하락 총영향 Σβ⁻ | 차이 Σβ⁻ − Σβ⁺ | β⁻ = β⁺ 동시 χ² (p) |
|---|---:|---:|---:|---:|---:|
| 미래 RS⁺ | 1 | 0.377*** (t 7.15, p <0.001) | 0.287*** (t 3.76, p <0.001) | -0.090 (t -0.81, p 0.416) | 11.4** (0.010) |
| 미래 RS⁺ | 5 | 0.308*** (t 6.32, p <0.001) | 0.266** (t 3.01, p 0.003) | -0.042 (t -0.34, p 0.734) | 5.4 (0.147) |
| 미래 RS⁺ | 22 | 0.203*** (t 5.78, p <0.001) | 0.208* (t 2.40, p 0.016) | 0.005 (t 0.04, p 0.967) | 2.2 (0.534) |
| 미래 RS⁻ | 1 | 0.362*** (t 5.53, p <0.001) | 0.324*** (t 4.07, p <0.001) | -0.038 (t -0.29, p 0.769) | 0.4 (0.946) |
| 미래 RS⁻ | 5 | 0.318*** (t 5.47, p <0.001) | 0.315*** (t 3.70, p <0.001) | -0.003 (t -0.02, p 0.981) | 1.8 (0.608) |
| 미래 RS⁻ | 22 | 0.241*** (t 4.85, p <0.001) | 0.296** (t 2.91, p 0.004) | 0.054 (t 0.38, p 0.704) | 4.5 (0.210) |
| 미래 RV | 1 | 0.738*** (t 7.80, p <0.001) | 0.611*** (t 4.33, p <0.001) | -0.127 (t -0.59, p 0.558) | 12.5** (0.006) |
| 미래 RV | 5 | 0.626*** (t 6.66, p <0.001) | 0.581*** (t 3.60, p <0.001) | -0.045 (t -0.19, p 0.850) | 4.3 (0.231) |
| 미래 RV | 22 | 0.444*** (t 5.33, p <0.001) | 0.503** (t 2.71, p 0.007) | 0.059 (t 0.23, p 0.817) | 3.5 (0.315) |

- **미국 종목**: 하락 총영향은 9개 모두 양(+)이고 유의하다 (p ≤ 0.030). 상승 총영향은 미래 RS⁻ 1일(0.208, p = 0.001)만 유의하다. 차이는 모두 양(+)이지만 유의하지 않다 (p = 0.10 ~ 0.41). 방향은 "하락 우위"지만, 상승 쪽과 통계적으로 구분할 만큼 크지는 않다.
- **한국 종목**: 상승·하락 총영향이 **둘 다 모든 조합에서 유의**하고(p ≤ 0.016) 크기가 비슷하다(예: 미래 RV 1일 0.738 vs 0.611). 차이는 모두 p ≥ 0.41이다. 장중 기준으로는 **대칭**이다.

### 2.4 같은 기간, 측정 방식별 비교 (목표: 미래 RV, 차이 Σβ⁻ − Σβ⁺)

| 시장 | 단위 | h=1: Σβ⁻ − Σβ⁺ (p) | h=5: Σβ⁻ − Σβ⁺ (p) | h=22: Σβ⁻ − Σβ⁺ (p) |
|---|---|---:|---:|---:|
| US | 지수 (장중) | 3.134 (0.158) | 1.682 (0.149) | 3.683 (0.183) |
| US | ETF (장중) | 3.159 (0.161) | 1.692 (0.156) | 3.628 (0.192) |
| US | 종목 패널 (장중) | 0.742 (0.191) | 0.622 (0.137) | 0.855 (0.113) |
| US | 지수 (일봉, 같은 기간) | 3.245* (0.017) | 2.924* (0.039) | 2.764 (0.089) |
| KR | 지수 (장중) | -4.346* (0.031) | -3.986 (0.090) | -1.682 (0.223) |
| KR | ETF (장중) | -3.896* (0.047) | -4.001 (0.129) | -2.625 (0.093) |
| KR | 종목 패널 (장중) | -0.127 (0.558) | -0.045 (0.850) | 0.059 (0.817) |
| KR | 지수 (일봉, 같은 기간) | 1.802 (0.225) | 1.159 (0.456) | -0.479 (0.731) |

- **미국**: 같은 3년을 일봉(종가 대 종가)으로 재면 하락 우위가 유의하다 (1일 3.25, p = 0.017 / 5일 2.92, p = 0.039). 장중 봉으로 재면 차이가 비슷한 크기로 추정되지만 유의하지 않다. 종목 패널은 차이가 작다.
- **한국**: 일봉으로 재면 하락 우위(양수)지만 유의하지 않다. 장중으로 재면 상승 우위(음수)로 뒤집힌다.
- 기간이 같으므로, 이 차이는 **측정 방식**, 그중에서도 장중 지표가 빼는 **밤사이 움직임** 때문일 가능성이 크다. 3장에서 직접 확인한다.

---

## 3. 장중 vs 밤사이 분리 (60분봉 3년)

장중 RS±(일·주·월) 6개와 밤사이 ON±(일·주·월) 6개를 **한 회귀에** 넣었다. 목표는 미래 장중 반분산이다. 두 블록 각각의 총영향과 방향 차이를 검정했다.

### 3.1 미국 지수

**장중 블록**

| 목표 | h | 상승 총영향 Σβ⁺ | 하락 총영향 Σβ⁻ | 차이 Σβ⁻ − Σβ⁺ | β⁻ = β⁺ 동시 χ² (p) |
|---|---:|---:|---:|---:|---:|
| 미래 RS⁺ | 1 | -0.463 (t -1.00, p 0.316) | 0.853 (t 1.71, p 0.087) | 1.315 (t 1.42, p 0.156) | 21.9*** (<0.001) |
| 미래 RS⁺ | 5 | -0.346 (t -1.19, p 0.234) | 0.291 (t 1.90, p 0.057) | 0.637 (t 1.71, p 0.088) | 34.6*** (<0.001) |
| 미래 RS⁺ | 22 | -0.624 (t -1.59, p 0.113) | 1.816 (t 1.63, p 0.103) | 2.440 (t 1.64, p 0.102) | 11.2* (0.011) |
| 미래 RS⁻ | 1 | 0.050 (t 0.26, p 0.798) | -0.020 (t -0.10, p 0.923) | -0.070 (t -0.24, p 0.811) | 8.5* (0.037) |
| 미래 RS⁻ | 5 | -0.049 (t -0.29, p 0.768) | 0.137 (t 0.88, p 0.379) | 0.186 (t 0.63, p 0.527) | 1.0 (0.813) |
| 미래 RS⁻ | 22 | -0.244 (t -1.10, p 0.269) | 0.848 (t 1.39, p 0.165) | 1.093 (t 1.33, p 0.184) | 1.8 (0.622) |
| 미래 RV | 1 | -0.413 (t -0.70, p 0.485) | 0.832* (t 2.08, p 0.038) | 1.245 (t 1.32, p 0.187) | 16.7*** (<0.001) |
| 미래 RV | 5 | -0.395 (t -0.90, p 0.369) | 0.428 (t 1.59, p 0.113) | 0.823 (t 1.33, p 0.184) | 13.9** (0.003) |
| 미래 RV | 22 | -0.868 (t -1.42, p 0.156) | 2.665 (t 1.55, p 0.122) | 3.533 (t 1.53, p 0.126) | 6.6 (0.085) |

**밤사이 블록**

| 목표 | h | 상승 총영향 Σβ⁺ | 하락 총영향 Σβ⁻ | 차이 Σβ⁻ − Σβ⁺ | β⁻ = β⁺ 동시 χ² (p) |
|---|---:|---:|---:|---:|---:|
| 미래 RS⁺ | 1 | 0.658* (t 2.03, p 0.043) | 0.367 (t 1.81, p 0.070) | -0.291 (t -0.88, p 0.378) | 4.7 (0.196) |
| 미래 RS⁺ | 5 | -0.090 (t -0.53, p 0.598) | 0.852* (t 2.25, p 0.025) | 0.943* (t 2.16, p 0.031) | 6.7 (0.080) |
| 미래 RS⁺ | 22 | -0.737 (t -1.68, p 0.094) | 0.251 (t 1.96, p 0.050) | 0.988* (t 2.11, p 0.035) | 9.0* (0.029) |
| 미래 RS⁻ | 1 | -0.345 (t -1.77, p 0.077) | 0.638* (t 2.07, p 0.039) | 0.984* (t 2.21, p 0.027) | 7.5 (0.058) |
| 미래 RS⁻ | 5 | -0.230 (t -1.39, p 0.164) | 0.456** (t 2.59, p 0.009) | 0.686* (t 2.43, p 0.015) | 17.6*** (<0.001) |
| 미래 RS⁻ | 22 | -0.412 (t -1.69, p 0.090) | 0.156* (t 2.10, p 0.035) | 0.568* (t 2.18, p 0.029) | 16.9*** (<0.001) |
| 미래 RV | 1 | 0.312 (t 1.31, p 0.190) | 1.005* (t 2.33, p 0.020) | 0.693* (t 2.16, p 0.031) | 5.4 (0.146) |
| 미래 RV | 5 | -0.321 (t -1.02, p 0.306) | 1.308* (t 2.36, p 0.018) | 1.629* (t 2.31, p 0.021) | 9.6* (0.022) |
| 미래 RV | 22 | -1.149 (t -1.69, p 0.092) | 0.407* (t 2.05, p 0.040) | 1.556* (t 2.14, p 0.032) | 11.4** (0.010) |

- **밤사이 하락**이 이후 장중 변동성을 유의하게 키운다. 미래 RV 1일 1.005 (p = 0.020), 5일 1.308 (p = 0.018), 22일 0.407 (p = 0.040)이다.
- 밤사이의 **하락 − 상승 차이도 유의**하다. 미래 RV p = 0.031 / 0.021 / 0.032, 미래 RS⁻ p = 0.027 / 0.015 / 0.029다.
- 밤사이를 넣은 뒤에는 장중 블록의 하락 − 상승 차이가 모든 조합에서 유의하지 않다 (p ≥ 0.088).
- **결론: 미국 지수의 "하락이 더 키운다"는 밤사이 하락(갭 하락)이 이끈다.**

### 3.2 미국 종목 패널

**장중 블록**

| 목표 | h | 상승 총영향 Σβ⁺ | 하락 총영향 Σβ⁻ | 차이 Σβ⁻ − Σβ⁺ | β⁻ = β⁺ 동시 χ² (p) |
|---|---:|---:|---:|---:|---:|
| 미래 RS⁺ | 1 | 0.025 (t 0.18, p 0.854) | 0.656* (t 2.08, p 0.038) | 0.631 (t 1.42, p 0.157) | 3.4 (0.330) |
| 미래 RS⁺ | 5 | 0.037 (t 0.35, p 0.724) | 0.427** (t 2.59, p 0.009) | 0.390 (t 1.56, p 0.119) | 2.9 (0.413) |
| 미래 RS⁺ | 22 | -0.080 (t -0.53, p 0.595) | 0.437* (t 2.09, p 0.037) | 0.516 (t 1.54, p 0.125) | 2.7 (0.448) |
| 미래 RS⁻ | 1 | 0.195*** (t 3.41, p <0.001) | 0.287*** (t 4.03, p <0.001) | 0.092 (t 0.80, p 0.423) | 27.5*** (<0.001) |
| 미래 RS⁻ | 5 | 0.085 (t 1.06, p 0.290) | 0.297*** (t 3.31, p <0.001) | 0.212 (t 1.41, p 0.160) | 3.4 (0.332) |
| 미래 RS⁻ | 22 | -0.031 (t -0.28, p 0.778) | 0.302** (t 2.61, p 0.009) | 0.333 (t 1.63, p 0.102) | 3.3 (0.346) |
| 미래 RV | 1 | 0.219 (t 1.20, p 0.229) | 0.942* (t 2.51, p 0.012) | 0.723 (t 1.33, p 0.185) | 6.3 (0.096) |
| 미래 RV | 5 | 0.122 (t 0.67, p 0.506) | 0.724** (t 2.90, p 0.004) | 0.602 (t 1.53, p 0.127) | 2.9 (0.400) |
| 미래 RV | 22 | -0.111 (t -0.43, p 0.669) | 0.739* (t 2.29, p 0.022) | 0.849 (t 1.58, p 0.114) | 2.6 (0.451) |

**밤사이 블록**

| 목표 | h | 상승 총영향 Σβ⁺ | 하락 총영향 Σβ⁻ | 차이 Σβ⁻ − Σβ⁺ | β⁻ = β⁺ 동시 χ² (p) |
|---|---:|---:|---:|---:|---:|
| 미래 RS⁺ | 1 | 0.000 (t 0.04, p 0.969) | 0.044** (t 3.14, p 0.002) | 0.044 (t 1.93, p 0.054) | 8.0* (0.046) |
| 미래 RS⁺ | 5 | -0.006 (t -0.38, p 0.707) | 0.039* (t 2.43, p 0.015) | 0.046 (t 1.46, p 0.145) | 2.5 (0.471) |
| 미래 RS⁺ | 22 | -0.005 (t -0.28, p 0.778) | 0.022* (t 2.56, p 0.011) | 0.027 (t 1.22, p 0.221) | 3.1 (0.381) |
| 미래 RS⁻ | 1 | 0.007 (t 0.83, p 0.405) | 0.014 (t 1.83, p 0.067) | 0.007 (t 0.54, p 0.590) | 4.5 (0.215) |
| 미래 RS⁻ | 5 | 0.005 (t 0.50, p 0.614) | 0.014* (t 2.02, p 0.043) | 0.009 (t 0.67, p 0.501) | 1.8 (0.625) |
| 미래 RS⁻ | 22 | 0.011 (t 0.88, p 0.380) | 0.011 (t 1.78, p 0.074) | -0.000 (t -0.03, p 0.976) | 6.1 (0.107) |
| 미래 RV | 1 | 0.008 (t 0.48, p 0.632) | 0.058** (t 3.03, p 0.002) | 0.051 (t 1.55, p 0.122) | 3.7 (0.298) |
| 미래 RV | 5 | -0.001 (t -0.03, p 0.974) | 0.054* (t 2.47, p 0.014) | 0.055 (t 1.26, p 0.209) | 1.7 (0.646) |
| 미래 RV | 22 | 0.006 (t 0.21, p 0.831) | 0.032* (t 2.35, p 0.019) | 0.026 (t 0.78, p 0.433) | 5.1 (0.163) |

- 장중 하락(미래 RV 1일 0.942, p = 0.012)과 밤사이 하락(0.058, p = 0.002)이 둘 다 유의하다. 상승 쪽은 장중 미래 RS⁻ 1일(0.195, p = 0.001) 외에는 두 블록 모두 유의하지 않다.
- 방향 차이는 두 블록 모두 유의하지 않다 (밤사이 미래 RS⁺ 1일 p = 0.054가 가장 작다).
- 밤사이 계수의 절대 크기가 작은 것은 종목의 밤사이 수익률 제곱이 장중 합계보다 작은 단위라서다.

### 3.3 한국 지수

**장중 블록**

| 목표 | h | 상승 총영향 Σβ⁺ | 하락 총영향 Σβ⁻ | 차이 Σβ⁻ − Σβ⁺ | β⁻ = β⁺ 동시 χ² (p) |
|---|---:|---:|---:|---:|---:|
| 미래 RS⁺ | 1 | -0.049 (t -0.06, p 0.955) | 0.607 (t 0.86, p 0.389) | 0.656 (t 0.43, p 0.668) | 5.4 (0.146) |
| 미래 RS⁺ | 5 | 0.299 (t 0.41, p 0.679) | 0.574 (t 0.88, p 0.379) | 0.275 (t 0.20, p 0.838) | 4.8 (0.190) |
| 미래 RS⁺ | 22 | 1.167** (t 2.92, p 0.003) | -0.340 (t -0.91, p 0.365) | -1.508* (t -1.96, p 0.050) | 4.8 (0.184) |
| 미래 RS⁻ | 1 | 3.238** (t 2.80, p 0.005) | -2.625* (t -2.22, p 0.026) | -5.863* (t -2.53, p 0.011) | 11.4** (0.010) |
| 미래 RS⁻ | 5 | 2.972** (t 3.20, p 0.001) | -1.927* (t -2.22, p 0.027) | -4.899** (t -2.75, p 0.006) | 16.8*** (<0.001) |
| 미래 RS⁻ | 22 | 1.592** (t 3.16, p 0.002) | -0.397 (t -0.86, p 0.390) | -1.989* (t -2.07, p 0.038) | 9.2* (0.027) |
| 미래 RV | 1 | 3.189* (t 2.43, p 0.015) | -2.018 (t -1.60, p 0.109) | -5.207* (t -2.08, p 0.037) | 8.8* (0.031) |
| 미래 RV | 5 | 3.270* (t 2.51, p 0.012) | -1.353 (t -1.17, p 0.242) | -4.624 (t -1.90, p 0.058) | 3.6 (0.308) |
| 미래 RV | 22 | 2.759** (t 3.13, p 0.002) | -0.737 (t -0.90, p 0.368) | -3.496* (t -2.07, p 0.039) | 5.7 (0.130) |

**밤사이 블록**

| 목표 | h | 상승 총영향 Σβ⁺ | 하락 총영향 Σβ⁻ | 차이 Σβ⁻ − Σβ⁺ | β⁻ = β⁺ 동시 χ² (p) |
|---|---:|---:|---:|---:|---:|
| 미래 RS⁺ | 1 | -0.029 (t -0.22, p 0.822) | 0.110 (t 0.52, p 0.606) | 0.139 (t 0.48, p 0.628) | 10.9* (0.012) |
| 미래 RS⁺ | 5 | -0.071 (t -0.61, p 0.541) | -0.031 (t -0.17, p 0.862) | 0.040 (t 0.15, p 0.880) | 9.8* (0.020) |
| 미래 RS⁺ | 22 | 0.184 (t 1.12, p 0.263) | -0.204* (t -2.54, p 0.011) | -0.388 (t -1.75, p 0.080) | 12.7** (0.005) |
| 미래 RS⁻ | 1 | 0.154 (t 0.59, p 0.555) | 0.584* (t 2.03, p 0.043) | 0.430 (t 0.92, p 0.358) | 5.9 (0.114) |
| 미래 RS⁻ | 5 | 0.098 (t 0.39, p 0.695) | 0.270 (t 1.22, p 0.223) | 0.172 (t 0.42, p 0.676) | 12.6** (0.005) |
| 미래 RS⁻ | 22 | 0.211 (t 1.04, p 0.299) | -0.274** (t -2.83, p 0.005) | -0.485 (t -1.71, p 0.088) | 21.1*** (<0.001) |
| 미래 RV | 1 | 0.124 (t 0.42, p 0.678) | 0.693 (t 1.56, p 0.119) | 0.569 (t 0.90, p 0.369) | 6.4 (0.092) |
| 미래 RV | 5 | 0.027 (t 0.08, p 0.937) | 0.239 (t 0.66, p 0.507) | 0.212 (t 0.33, p 0.740) | 9.2* (0.026) |
| 미래 RV | 22 | 0.395 (t 1.08, p 0.279) | -0.477** (t -2.76, p 0.006) | -0.873 (t -1.73, p 0.083) | 17.0*** (<0.001) |

- 밤사이를 넣어도 **장중 상승 우위**는 그대로다 (미래 RV 1일 Σβ⁺ 3.19, p = 0.015 / 차이 −5.21, p = 0.037).
- 밤사이 하락은 단기(미래 RS⁻ 1일 0.584, p = 0.043)에는 양(+)이다. 22일 지평에서는 음(−)이다 (미래 RV −0.477, p = 0.006).
- 2023~2026년 한국은 밤사이를 분리해도 일봉에서 본 비대칭이 나타나지 않는다. 1.6절의 극단적 국면에서는 큰 상승일이 이후의 큰 하락을 예고했다. 이 표본에서 가장 두드러진 패턴이다.

### 3.4 한국 종목 패널

**장중 블록**

| 목표 | h | 상승 총영향 Σβ⁺ | 하락 총영향 Σβ⁻ | 차이 Σβ⁻ − Σβ⁺ | β⁻ = β⁺ 동시 χ² (p) |
|---|---:|---:|---:|---:|---:|
| 미래 RS⁺ | 1 | 0.332*** (t 6.33, p <0.001) | 0.237** (t 3.19, p 0.001) | -0.095 (t -0.87, p 0.386) | 10.9* (0.012) |
| 미래 RS⁺ | 5 | 0.273*** (t 5.60, p <0.001) | 0.219** (t 2.61, p 0.009) | -0.054 (t -0.45, p 0.655) | 5.4 (0.146) |
| 미래 RS⁺ | 22 | 0.160*** (t 4.12, p <0.001) | 0.156* (t 1.99, p 0.046) | -0.004 (t -0.04, p 0.971) | 1.6 (0.671) |
| 미래 RS⁻ | 1 | 0.307*** (t 5.45, p <0.001) | 0.270** (t 2.96, p 0.003) | -0.037 (t -0.30, p 0.765) | 0.4 (0.949) |
| 미래 RS⁻ | 5 | 0.276*** (t 4.80, p <0.001) | 0.271** (t 2.92, p 0.004) | -0.005 (t -0.04, p 0.966) | 1.8 (0.625) |
| 미래 RS⁻ | 22 | 0.201*** (t 4.21, p <0.001) | 0.251* (t 2.37, p 0.018) | 0.050 (t 0.36, p 0.722) | 3.9 (0.275) |
| 미래 RV | 1 | 0.639*** (t 7.87, p <0.001) | 0.507** (t 3.28, p 0.001) | -0.132 (t -0.63, p 0.530) | 11.7** (0.009) |
| 미래 RV | 5 | 0.548*** (t 6.13, p <0.001) | 0.490** (t 2.94, p 0.003) | -0.059 (t -0.26, p 0.798) | 4.4 (0.221) |
| 미래 RV | 22 | 0.361*** (t 4.30, p <0.001) | 0.407* (t 2.24, p 0.025) | 0.046 (t 0.19, p 0.853) | 2.9 (0.406) |

**밤사이 블록**

| 목표 | h | 상승 총영향 Σβ⁺ | 하락 총영향 Σβ⁻ | 차이 Σβ⁻ − Σβ⁺ | β⁻ = β⁺ 동시 χ² (p) |
|---|---:|---:|---:|---:|---:|
| 미래 RS⁺ | 1 | 0.043 (t 1.50, p 0.134) | 0.001 (t 0.05, p 0.958) | -0.042 (t -0.87, p 0.382) | 3.9 (0.276) |
| 미래 RS⁺ | 5 | 0.049 (t 1.61, p 0.107) | -0.014 (t -0.58, p 0.564) | -0.062 (t -1.26, p 0.207) | 6.3 (0.099) |
| 미래 RS⁺ | 22 | 0.053 (t 1.76, p 0.079) | -0.014 (t -0.74, p 0.462) | -0.067 (t -1.42, p 0.155) | 4.0 (0.265) |
| 미래 RS⁻ | 1 | 0.030 (t 1.26, p 0.206) | 0.029 (t 1.22, p 0.223) | -0.001 (t -0.02, p 0.982) | 0.1 (0.989) |
| 미래 RS⁻ | 5 | 0.027 (t 1.24, p 0.216) | 0.019 (t 1.01, p 0.315) | -0.008 (t -0.24, p 0.810) | 2.0 (0.566) |
| 미래 RS⁻ | 22 | 0.036 (t 1.42, p 0.155) | 0.005 (t 0.32, p 0.749) | -0.031 (t -0.84, p 0.402) | 3.2 (0.355) |
| 미래 RV | 1 | 0.073 (t 1.64, p 0.102) | 0.031 (t 0.75, p 0.453) | -0.043 (t -0.57, p 0.566) | 1.2 (0.764) |
| 미래 RV | 5 | 0.076 (t 1.62, p 0.106) | 0.005 (t 0.15, p 0.884) | -0.070 (t -0.92, p 0.356) | 6.3 (0.098) |
| 미래 RV | 22 | 0.089 (t 1.67, p 0.095) | -0.009 (t -0.28, p 0.778) | -0.098 (t -1.19, p 0.233) | 4.4 (0.221) |

- 장중 상승·하락이 **대칭적으로** 유의하다 (미래 RV 1일 0.639 vs 0.507, 차이 p = 0.530).
- 밤사이 블록은 어느 쪽도 유의하지 않다.

---

## 4. 5분봉·10분봉 60일 결과 (예비)

60거래일, 설명변수는 일·주 두 가지, 지평은 1·5일이다.

### 4.1 지수

#### 미국 5분봉

| 목표 | h | 상승 총영향 Σβ⁺ | 하락 총영향 Σβ⁻ | 차이 Σβ⁻ − Σβ⁺ | β⁻ = β⁺ 동시 χ² (p) |
|---|---:|---:|---:|---:|---:|
| 미래 RS⁺ | 1 | 1.590* (t 2.50, p 0.013) | -0.792 (t -1.78, p 0.075) | -2.383* (t -2.21, p 0.027) | 7.2* (0.027) |
| 미래 RS⁺ | 5 | 1.828** (t 2.65, p 0.008) | -1.110* (t -2.06, p 0.039) | -2.938* (t -2.40, p 0.017) | 7.6* (0.022) |
| 미래 RS⁻ | 1 | 2.940** (t 2.79, p 0.005) | -1.771* (t -2.28, p 0.023) | -4.711** (t -2.59, p 0.009) | 9.3** (0.010) |
| 미래 RS⁻ | 5 | 2.341** (t 2.99, p 0.003) | -1.463* (t -2.45, p 0.014) | -3.804** (t -2.76, p 0.006) | 9.0* (0.011) |
| 미래 RV | 1 | 4.530** (t 2.75, p 0.006) | -2.564* (t -2.17, p 0.030) | -7.094* (t -2.52, p 0.012) | 8.6* (0.014) |
| 미래 RV | 5 | 4.169** (t 2.87, p 0.004) | -2.573* (t -2.30, p 0.022) | -6.742** (t -2.63, p 0.009) | 8.6* (0.013) |

#### 미국 10분봉

| 목표 | h | 상승 총영향 Σβ⁺ | 하락 총영향 Σβ⁻ | 차이 Σβ⁻ − Σβ⁺ | β⁻ = β⁺ 동시 χ² (p) |
|---|---:|---:|---:|---:|---:|
| 미래 RS⁺ | 1 | 1.008** (t 2.80, p 0.005) | -0.255 (t -1.10, p 0.269) | -1.263* (t -2.19, p 0.029) | 9.3** (0.009) |
| 미래 RS⁺ | 5 | 1.437** (t 2.68, p 0.007) | -0.787* (t -2.09, p 0.036) | -2.223* (t -2.45, p 0.014) | 8.5* (0.014) |
| 미래 RS⁻ | 1 | 1.659** (t 2.60, p 0.009) | -0.736 (t -1.76, p 0.078) | -2.395* (t -2.34, p 0.019) | 5.6 (0.060) |
| 미래 RS⁻ | 5 | 1.986** (t 2.76, p 0.006) | -1.214* (t -2.41, p 0.016) | -3.201** (t -2.62, p 0.009) | 8.3* (0.016) |
| 미래 RV | 1 | 2.666** (t 2.78, p 0.005) | -0.992 (t -1.67, p 0.095) | -3.658* (t -2.42, p 0.016) | 6.5* (0.040) |
| 미래 RV | 5 | 3.423** (t 2.75, p 0.006) | -2.001* (t -2.31, p 0.021) | -5.424* (t -2.57, p 0.010) | 8.5* (0.014) |

#### 한국 5분봉

| 목표 | h | 상승 총영향 Σβ⁺ | 하락 총영향 Σβ⁻ | 차이 Σβ⁻ − Σβ⁺ | β⁻ = β⁺ 동시 χ² (p) |
|---|---:|---:|---:|---:|---:|
| 미래 RS⁺ | 1 | -0.687 (t -1.41, p 0.158) | 1.285** (t 2.68, p 0.007) | 1.972* (t 2.07, p 0.039) | 24.6*** (<0.001) |
| 미래 RS⁺ | 5 | 1.036** (t 3.13, p 0.002) | -0.465 (t -1.31, p 0.190) | -1.501* (t -2.23, p 0.026) | 25.8*** (<0.001) |
| 미래 RS⁻ | 1 | -0.647 (t -0.91, p 0.364) | 1.267* (t 1.96, p 0.050) | 1.914 (t 1.42, p 0.157) | 28.1*** (<0.001) |
| 미래 RS⁻ | 5 | 1.109*** (t 3.47, p <0.001) | -0.467 (t -1.34, p 0.181) | -1.576* (t -2.40, p 0.016) | 18.8*** (<0.001) |
| 미래 RV | 1 | -1.334 (t -1.14, p 0.254) | 2.552* (t 2.34, p 0.019) | 3.886 (t 1.73, p 0.083) | 34.0*** (<0.001) |
| 미래 RV | 5 | 2.145*** (t 3.32, p <0.001) | -0.932 (t -1.33, p 0.183) | -3.077* (t -2.32, p 0.020) | 21.6*** (<0.001) |

#### 한국 10분봉

| 목표 | h | 상승 총영향 Σβ⁺ | 하락 총영향 Σβ⁻ | 차이 Σβ⁻ − Σβ⁺ | β⁻ = β⁺ 동시 χ² (p) |
|---|---:|---:|---:|---:|---:|
| 미래 RS⁺ | 1 | -1.111* (t -2.09, p 0.036) | 1.667*** (t 3.52, p <0.001) | 2.778** (t 2.78, p 0.005) | 10.8** (0.004) |
| 미래 RS⁺ | 5 | 0.280 (t 0.88, p 0.381) | 0.121 (t 0.34, p 0.732) | -0.159 (t -0.25, p 0.806) | 12.3** (0.002) |
| 미래 RS⁻ | 1 | -0.668 (t -0.98, p 0.325) | 1.198* (t 2.00, p 0.045) | 1.866 (t 1.47, p 0.142) | 9.4** (0.009) |
| 미래 RS⁻ | 5 | 0.628 (t 1.52, p 0.128) | -0.135 (t -0.31, p 0.754) | -0.763 (t -0.93, p 0.352) | 7.4* (0.024) |
| 미래 RV | 1 | -1.778 (t -1.51, p 0.131) | 2.866** (t 2.76, p 0.006) | 4.644* (t 2.11, p 0.035) | 10.5** (0.005) |
| 미래 RV | 5 | 0.908 (t 1.26, p 0.208) | -0.014 (t -0.02, p 0.986) | -0.922 (t -0.64, p 0.525) | 9.6** (0.008) |

### 4.2 종목 패널

#### 미국 5분봉

| 목표 | h | 상승 총영향 Σβ⁺ | 하락 총영향 Σβ⁻ | 차이 Σβ⁻ − Σβ⁺ | β⁻ = β⁺ 동시 χ² (p) |
|---|---:|---:|---:|---:|---:|
| 미래 RS⁺ | 1 | 0.275*** (t 7.01, p <0.001) | 0.119** (t 3.14, p 0.002) | -0.157* (t -2.57, p 0.010) | 12.5** (0.002) |
| 미래 RS⁺ | 5 | 0.181*** (t 3.87, p <0.001) | 0.042 (t 0.86, p 0.391) | -0.139 (t -1.65, p 0.098) | 3.6 (0.167) |
| 미래 RS⁻ | 1 | 0.271*** (t 3.89, p <0.001) | 0.164** (t 2.93, p 0.003) | -0.107 (t -0.96, p 0.335) | 2.8 (0.247) |
| 미래 RS⁻ | 5 | 0.175** (t 3.28, p 0.001) | 0.057 (t 1.06, p 0.287) | -0.118 (t -1.20, p 0.231) | 1.4 (0.488) |
| 미래 RV | 1 | 0.546*** (t 6.14, p <0.001) | 0.283*** (t 3.65, p <0.001) | -0.264* (t -1.96, p 0.050) | 3.9 (0.144) |
| 미래 RV | 5 | 0.356*** (t 3.86, p <0.001) | 0.099 (t 1.06, p 0.287) | -0.257 (t -1.55, p 0.122) | 2.5 (0.289) |

#### 미국 10분봉

| 목표 | h | 상승 총영향 Σβ⁺ | 하락 총영향 Σβ⁻ | 차이 Σβ⁻ − Σβ⁺ | β⁻ = β⁺ 동시 χ² (p) |
|---|---:|---:|---:|---:|---:|
| 미래 RS⁺ | 1 | 0.153*** (t 4.05, p <0.001) | 0.155*** (t 3.37, p <0.001) | 0.002 (t 0.04, p 0.971) | 0.2 (0.921) |
| 미래 RS⁺ | 5 | 0.080* (t 2.28, p 0.022) | 0.073* (t 2.19, p 0.028) | -0.006 (t -0.14, p 0.889) | 0.0 (0.989) |
| 미래 RS⁻ | 1 | 0.277*** (t 4.06, p <0.001) | 0.125* (t 2.54, p 0.011) | -0.152 (t -1.55, p 0.121) | 3.1 (0.208) |
| 미래 RS⁻ | 5 | 0.120** (t 2.68, p 0.007) | 0.079 (t 1.82, p 0.069) | -0.041 (t -0.61, p 0.540) | 1.6 (0.442) |
| 미래 RV | 1 | 0.430*** (t 4.78, p <0.001) | 0.280*** (t 3.40, p <0.001) | -0.150 (t -1.20, p 0.230) | 1.7 (0.425) |
| 미래 RV | 5 | 0.200** (t 2.82, p 0.005) | 0.152* (t 2.20, p 0.028) | -0.047 (t -0.50, p 0.614) | 0.7 (0.712) |

#### 한국 5분봉

| 목표 | h | 상승 총영향 Σβ⁺ | 하락 총영향 Σβ⁻ | 차이 Σβ⁻ − Σβ⁺ | β⁻ = β⁺ 동시 χ² (p) |
|---|---:|---:|---:|---:|---:|
| 미래 RS⁺ | 1 | 0.162 (t 1.16, p 0.247) | 0.509*** (t 4.14, p <0.001) | 0.346 (t 1.39, p 0.164) | 5.2 (0.073) |
| 미래 RS⁺ | 5 | 0.350** (t 2.98, p 0.003) | 0.138** (t 2.85, p 0.004) | -0.212 (t -1.89, p 0.059) | 10.9** (0.004) |
| 미래 RS⁻ | 1 | 0.296* (t 2.29, p 0.022) | 0.333** (t 3.04, p 0.002) | 0.037 (t 0.17, p 0.866) | 9.9** (0.007) |
| 미래 RS⁻ | 5 | 0.377** (t 3.01, p 0.003) | 0.105 (t 1.53, p 0.127) | -0.271* (t -2.00, p 0.046) | 7.7* (0.021) |
| 미래 RV | 1 | 0.458 (t 1.73, p 0.083) | 0.841*** (t 3.76, p <0.001) | 0.383 (t 0.84, p 0.400) | 7.5* (0.023) |
| 미래 RV | 5 | 0.727** (t 3.05, p 0.002) | 0.243* (t 2.13, p 0.033) | -0.483* (t -2.02, p 0.043) | 9.5** (0.009) |

#### 한국 10분봉

| 목표 | h | 상승 총영향 Σβ⁺ | 하락 총영향 Σβ⁻ | 차이 Σβ⁻ − Σβ⁺ | β⁻ = β⁺ 동시 χ² (p) |
|---|---:|---:|---:|---:|---:|
| 미래 RS⁺ | 1 | 0.341** (t 2.95, p 0.003) | 0.289** (t 2.93, p 0.003) | -0.052 (t -0.26, p 0.791) | 9.6** (0.008) |
| 미래 RS⁺ | 5 | 0.431*** (t 4.67, p <0.001) | -0.009 (t -0.13, p 0.899) | -0.440*** (t -4.43, p <0.001) | 21.7*** (<0.001) |
| 미래 RS⁻ | 1 | 0.373*** (t 4.33, p <0.001) | 0.174* (t 2.14, p 0.032) | -0.199 (t -1.44, p 0.150) | 15.5*** (<0.001) |
| 미래 RS⁻ | 5 | 0.490*** (t 6.03, p <0.001) | -0.068 (t -0.71, p 0.477) | -0.558*** (t -4.59, p <0.001) | 21.2*** (<0.001) |
| 미래 RV | 1 | 0.714*** (t 3.98, p <0.001) | 0.463** (t 3.10, p 0.002) | -0.251 (t -0.90, p 0.370) | 17.9*** (<0.001) |
| 미래 RV | 5 | 0.921*** (t 5.48, p <0.001) | -0.077 (t -0.49, p 0.628) | -0.998*** (t -4.95, p <0.001) | 24.6*** (<0.001) |

### 4.3 측정 방식별 비교 (목표: 미래 RV, 차이 Σβ⁻ − Σβ⁺)

**5분봉 구간**

| 시장 | 단위 | h=1: Σβ⁻ − Σβ⁺ (p) | h=5: Σβ⁻ − Σβ⁺ (p) |
|---|---|---:|---:|
| US | 지수 (장중) | -7.094* (0.012) | -6.742** (0.009) |
| US | ETF (장중) | -7.163** (0.009) | -6.509* (0.013) |
| US | 종목 패널 (장중) | -0.264* (0.050) | -0.257 (0.122) |
| US | 지수 (일봉, 같은 기간) | 2.695** (0.006) | 2.137** (0.008) |
| KR | 지수 (장중) | 3.886 (0.083) | -3.077* (0.020) |
| KR | ETF (장중) | 0.463 (0.886) | -2.463 (0.486) |
| KR | 종목 패널 (장중) | 0.383 (0.400) | -0.483* (0.043) |
| KR | 지수 (일봉, 같은 기간) | 2.392* (0.018) | 1.336*** (<0.001) |

**10분봉 구간** (같은 60일, 일봉 대조군은 위와 같음)

| 시장 | 단위 | h=1: Σβ⁻ − Σβ⁺ (p) | h=5: Σβ⁻ − Σβ⁺ (p) |
|---|---|---:|---:|
| US | 지수 (장중) | -3.658* (0.016) | -5.424* (0.010) |
| US | ETF (장중) | -3.668* (0.011) | -5.149* (0.038) |
| US | 종목 패널 (장중) | -0.150 (0.230) | -0.047 (0.614) |
| US | 지수 (일봉, 같은 기간) | 2.695** (0.006) | 2.137** (0.008) |
| KR | 지수 (장중) | 4.644* (0.035) | -0.922 (0.525) |
| KR | ETF (장중) | 4.213* (0.015) | 0.326 (0.768) |
| KR | 종목 패널 (장중) | -0.251 (0.370) | -0.998*** (<0.001) |
| KR | 지수 (일봉, 같은 기간) | 2.392* (0.018) | 1.336*** (<0.001) |

### 4.4 해석

- **미국 지수 (5분·10분 모두)**: 장중 **상승** 변동성이 이후 변동성을 키우고(5분봉 미래 RV 1일 Σβ⁺ 4.53, p = 0.006), 하락 쪽은 음(−)이다. 그래서 차이가 음(−)으로 유의하다 (5분 −7.09, p = 0.012 / 10분 −3.66, p = 0.016). 그러나 **같은 60일의 일봉 대조군은 정반대로 하락 우위가 유의**하다 (2.70, p = 0.006). 이 60일은 ±3% 초과일이 하루도 없는 매우 조용한 기간이다. 그래서 몇 번의 작은 장중 반등이 결과를 좌우했을 가능성이 크다.
- **한국 지수**: 1일 지평은 하락 우위(5분 1.97, p = 0.039 / 10분 2.78, p = 0.005, 미래 RS⁺ 기준)다. 5일 지평은 상승 우위(5분 미래 RV −3.08, p = 0.020)로 **지평에 따라 뒤집힌다.** 이 60일에는 ±8% 초과일이 4번 있어서 몇 개의 극단일이 결과를 지배한다.
- **종목 패널**: 미국은 상승·하락 모두 양(+)이다. 5분봉 1일에서는 상승이 조금 더 크고(미래 RS⁺ 차이 −0.157, p = 0.010), 10분봉에서는 차이가 없다. 한국은 5일 지평에서 상승 우위가 나오는데, 10분봉에서 더 강하다 (미래 RV 차이 −0.998, p < 0.001).
- **5분과 10분처럼 봉 간격만 바꿔도 유의성과 부호가 달라지는 경우가 많다.** 60일 표본으로는 장중 비대칭에 대해 안정적인 결론을 내릴 수 없다.

---

## 5. 종합 결론

| 질문 | 60분봉 3년 (주 분석) | 5·10분봉 60일 (예비) |
|---|---|---|
| 오늘 장중 **하락** 변동성이 이후 변동성을 키우는가 | **예** — 미국 종목(모든 조합 유의), 한국 종목(모든 조합 유의). 미국 지수는 방향만 같고 경계 수준. 한국 지수는 오히려 음(−) 또는 0 | 혼재 |
| 오늘 장중 **상승** 변동성이 이후 변동성을 키우는가 | 한국(지수·종목) **예**. 미국 종목은 대체로 아니오, 미국 지수는 아니오 | 미국 지수·종목 예, 한국 혼재 |
| 장중 하락의 영향이 상승보다 큰가 | **통계적으로 구분되지 않음** (미국 종목·지수 방향은 +, 한국 지수는 오히려 −) | 봉 간격·지평에 따라 부호가 바뀜 |
| 일봉에서 본 하락 우위는 어디서 오는가 | 미국 지수: **밤사이 하락**에서 유의 (장중 차이는 유의하지 않음) | — |

**경제적 해석.**
1. 미국에서 "하락이 변동성을 더 키운다"는 효과는 **장이 닫혀 있는 동안의 하락**, 즉 시가 갭 하락에 집중돼 있다. 밤사이 악재는 장중에 조금씩 반영되는 하락보다 정보 충격의 성격이 강해서, 이후 변동성을 더 오래 키운다고 해석할 수 있다.
2. 장중에는 상승이든 하락이든 **움직임의 크기 자체**가 이후 변동성을 예고한다. 방향에 따른 차이는 작다. 한국 종목에서 특히 대칭적이다.
3. 한국 지수 2023~2026은 표본 자체가 이례적이다(3년 +182%, 급등 후 급락). 이 기간의 "상승 변동성 → 이후 하락 변동성" 패턴은 과열 국면의 급등이 뒤따르는 급락을 예고한 것으로 볼 수 있다. 다만 다른 기간으로 일반화하기는 어렵다.

**일봉 보고서(`REPORT_semivariance.md`)와의 관계.**
- 일봉(2005~2020): 과거 하락 변동성의 총영향은 유의하고, 상승은 0이며, 하락 우위다.
- 이번 장중 분석은 그 비대칭이 **밤사이 움직임을 포함한 종가 대 종가 수익률에서** 나타난다는 것을 보여준다. 장중만 보면 약해지거나(미국) 사라진다(한국 종목).
- 즉 "하락 변동성" 효과를 말할 때는 **어떤 시간 구간의 하락인지**(장중 / 밤사이)를 구분해야 한다.

---

## 6. 한계

1. **표본 기간.** 60분봉은 약 3년이고, 5·10분봉은 60거래일뿐이다. yfinance가 5분봉은 60일, 60분봉은 약 730일까지만 준다. 5·10분봉 결론은 예비적이다. `step0_collect_intraday.py`를 매주 실행해 5분봉을 쌓으면 다시 검정할 수 있다.
2. **한국 장중 데이터가 장 마감 전에 끊긴다.** yfinance 한국 봉은 14:55(5분봉) / 14:00(60분봉) 봉이 마지막이다. 그래서 15:00 ~ 15:30(마지막 20분과 종가 동시호가)이 장중 지표에서 빠지고 "밤사이" 수익률에 섞인다. 한국의 장중/밤사이 구분은 그만큼 부정확하다.
3. **60분봉은 하루 수익률이 5~6개뿐**이라 장중 반분산의 측정오차가 크다.
4. **한국 표본 기간의 극단성.** 데이터상 3년 +182%, 하루 ±8% 초과 11회다. 외부 기록과 대조하지 않았다.
5. **종목 패널 정규화.** 종목별 평균 RV는 표본 전체로 계산했다. 이것은 단위를 맞추는 용도라서 예측 주장에는 영향이 없다. 종목 고정효과는 넣지 않았다(짧은 패널에서 편향이 생길 수 있어서).
6. **다중검정.** 봉 3종 × 단위 4종 × 시장 2 × 지평 × 목표로 검정 수가 많다. 결론은 여러 조합에서 반복되거나 대조군과 함께 해석되는 결과에만 기댔다.
7. 모든 장중 데이터는 테스트 구간(2021년 이후)이다. S1·S2S3 동결 후 실행했고 접근은 `logs/test_access.log`에 기록됐다.

---

## 부록 A. 개별 계수표 (β, t, p)

### A.1 60분봉 지수 — 미국


**h = 1일**

| 설명변수 | 미래 RS⁺ β | t | p | 미래 RS⁻ β | t | p | 미래 RV β | t | p |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 장중 RS⁺ 일 | 0.163*** | 3.89 | <0.001 | 0.227*** | 18.73 | <0.001 | 0.390*** | 8.16 | <0.001 |
| 장중 RS⁻ 일 | 1.346** | 2.69 | 0.007 | 0.049 | 1.04 | 0.298 | 1.395** | 2.92 | 0.004 |
| 장중 RS⁺ 주 | -0.279 | -1.33 | 0.185 | 0.022 | 0.33 | 0.740 | -0.257 | -1.05 | 0.295 |
| 장중 RS⁻ 주 | 0.227 | 0.91 | 0.364 | 0.064 | 0.50 | 0.621 | 0.291 | 1.05 | 0.293 |
| 장중 RS⁺ 월 | -0.586 | -1.03 | 0.303 | 0.014 | 0.13 | 0.900 | -0.572 | -0.89 | 0.374 |
| 장중 RS⁻ 월 | 0.928 | 1.10 | 0.273 | 0.081 | 0.34 | 0.730 | 1.008 | 0.98 | 0.328 |
| 적합도 | R² 0.444, n 699 | |  | R² 0.294, n 699 | |  | R² 0.485, n 699 | |  |

**h = 5일**

| 설명변수 | 미래 RS⁺ β | t | p | 미래 RS⁻ β | t | p | 미래 RV β | t | p |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 장중 RS⁺ 일 | 0.064*** | 4.81 | <0.001 | 0.057*** | 9.00 | <0.001 | 0.121*** | 6.58 | <0.001 |
| 장중 RS⁻ 일 | 0.469** | 2.75 | 0.006 | 0.134** | 3.26 | 0.001 | 0.603** | 2.90 | 0.004 |
| 장중 RS⁺ 주 | -0.022 | -0.41 | 0.679 | 0.060 | 1.78 | 0.074 | 0.038 | 0.51 | 0.608 |
| 장중 RS⁻ 주 | 0.036 | 0.34 | 0.737 | 0.023 | 0.26 | 0.794 | 0.059 | 0.33 | 0.743 |
| 장중 RS⁺ 월 | -0.294 | -1.13 | 0.257 | -0.049 | -0.34 | 0.733 | -0.343 | -0.88 | 0.379 |
| 장중 RS⁻ 월 | 0.650 | 1.39 | 0.164 | 0.185 | 0.63 | 0.526 | 0.835 | 1.14 | 0.256 |
| 적합도 | R² 0.220, n 695 | |  | R² 0.185, n 695 | |  | R² 0.213, n 695 | |  |

**h = 22일**

| 설명변수 | 미래 RS⁺ β | t | p | 미래 RS⁻ β | t | p | 미래 RV β | t | p |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 장중 RS⁺ 일 | 0.030** | 3.08 | 0.002 | 0.022*** | 4.50 | <0.001 | 0.052*** | 3.57 | <0.001 |
| 장중 RS⁻ 일 | 0.105*** | 3.51 | <0.001 | 0.041*** | 3.60 | <0.001 | 0.146*** | 3.64 | <0.001 |
| 장중 RS⁺ 주 | 0.112 | 1.41 | 0.159 | 0.072 | 1.81 | 0.070 | 0.184 | 1.55 | 0.120 |
| 장중 RS⁻ 주 | -0.208 | -1.06 | 0.291 | -0.114 | -1.16 | 0.247 | -0.322 | -1.10 | 0.273 |
| 장중 RS⁺ 월 | -0.931 | -1.33 | 0.184 | -0.400 | -1.08 | 0.281 | -1.331 | -1.24 | 0.213 |
| 장중 RS⁻ 월 | 1.890 | 1.39 | 0.163 | 0.873 | 1.21 | 0.226 | 2.763 | 1.33 | 0.183 |
| 적합도 | R² 0.214, n 678 | |  | R² 0.162, n 678 | |  | R² 0.198, n 678 | |  |

### A.2 60분봉 지수 — 한국


**h = 1일**

| 설명변수 | 미래 RS⁺ β | t | p | 미래 RS⁻ β | t | p | 미래 RV β | t | p |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 장중 RS⁺ 일 | 0.075 | 0.75 | 0.456 | -0.157 | -1.85 | 0.064 | -0.082 | -0.51 | 0.607 |
| 장중 RS⁻ 일 | 0.211*** | 3.49 | <0.001 | 0.149 | 1.55 | 0.121 | 0.360** | 2.61 | 0.009 |
| 장중 RS⁺ 주 | 0.158 | 1.23 | 0.220 | 0.136 | 0.64 | 0.520 | 0.294 | 1.07 | 0.282 |
| 장중 RS⁻ 주 | 0.093 | 0.74 | 0.459 | 0.128 | 0.72 | 0.471 | 0.221 | 1.69 | 0.090 |
| 장중 RS⁺ 월 | -0.044 | -0.05 | 0.958 | 3.205** | 3.11 | 0.002 | 3.161* | 2.18 | 0.029 |
| 장중 RS⁻ 월 | 0.224 | 0.35 | 0.724 | -1.778** | -2.77 | 0.006 | -1.554 | -1.64 | 0.101 |
| 적합도 | R² 0.320, n 706 | |  | R² 0.235, n 706 | |  | R² 0.361, n 706 | |  |

**h = 5일**

| 설명변수 | 미래 RS⁺ β | t | p | 미래 RS⁻ β | t | p | 미래 RV β | t | p |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 장중 RS⁺ 일 | 0.096 | 1.64 | 0.100 | 0.014 | 0.22 | 0.822 | 0.110 | 0.94 | 0.349 |
| 장중 RS⁻ 일 | 0.057 | 1.34 | 0.181 | 0.081*** | 3.51 | <0.001 | 0.138* | 2.51 | 0.012 |
| 장중 RS⁺ 주 | -0.093 | -1.16 | 0.245 | 0.030 | 0.24 | 0.807 | -0.063 | -0.34 | 0.730 |
| 장중 RS⁻ 주 | 0.191 | 1.40 | 0.162 | -0.073 | -0.47 | 0.637 | 0.118 | 0.45 | 0.652 |
| 장중 RS⁺ 월 | 0.348 | 0.40 | 0.688 | 2.715** | 3.28 | 0.001 | 3.063* | 2.09 | 0.037 |
| 장중 RS⁻ 월 | 0.099 | 0.17 | 0.865 | -1.232* | -2.46 | 0.014 | -1.132 | -1.21 | 0.227 |
| 적합도 | R² 0.438, n 702 | |  | R² 0.471, n 702 | |  | R² 0.500, n 702 | |  |

**h = 22일**

| 설명변수 | 미래 RS⁺ β | t | p | 미래 RS⁻ β | t | p | 미래 RV β | t | p |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 장중 RS⁺ 일 | 0.027 | 1.20 | 0.232 | 0.028 | 0.90 | 0.368 | 0.055 | 1.03 | 0.301 |
| 장중 RS⁻ 일 | 0.029* | 2.48 | 0.013 | 0.023 | 1.31 | 0.191 | 0.052 | 1.80 | 0.072 |
| 장중 RS⁺ 주 | 0.051 | 1.14 | 0.255 | 0.164** | 3.08 | 0.002 | 0.215* | 2.22 | 0.026 |
| 장중 RS⁻ 주 | 0.048 | 0.76 | 0.445 | -0.012 | -0.16 | 0.875 | 0.035 | 0.25 | 0.801 |
| 장중 RS⁺ 월 | 0.571* | 1.97 | 0.049 | 0.858* | 2.07 | 0.039 | 1.429* | 2.10 | 0.035 |
| 장중 RS⁻ 월 | -0.022 | -0.08 | 0.938 | -0.048 | -0.10 | 0.917 | -0.071 | -0.10 | 0.924 |
| 적합도 | R² 0.526, n 685 | |  | R² 0.550, n 685 | |  | R² 0.545, n 685 | |  |

### A.3 60분봉 종목 패널 — 미국


**h = 1일**

| 설명변수 | 미래 RS⁺ β | t | p | 미래 RS⁻ β | t | p | 미래 RV β | t | p |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 장중 RS⁺ 일 | 0.091*** | 5.61 | <0.001 | 0.178*** | 4.90 | <0.001 | 0.269*** | 5.56 | <0.001 |
| 장중 RS⁻ 일 | 0.351 | 1.47 | 0.142 | 0.039** | 2.84 | 0.004 | 0.390 | 1.64 | 0.101 |
| 장중 RS⁺ 주 | -0.030 | -0.28 | 0.783 | 0.032 | 0.66 | 0.511 | 0.002 | 0.01 | 0.991 |
| 장중 RS⁻ 주 | 0.256 | 1.92 | 0.055 | 0.083* | 2.19 | 0.028 | 0.339* | 2.16 | 0.031 |
| 장중 RS⁺ 월 | 0.003 | 0.05 | 0.964 | -0.002 | -0.03 | 0.974 | 0.001 | 0.00 | 0.996 |
| 장중 RS⁻ 월 | 0.098 | 1.56 | 0.119 | 0.186** | 2.78 | 0.005 | 0.284*** | 3.99 | <0.001 |
| 적합도 | R² 0.132, n 34943 | |  | R² 0.090, n 34943 | |  | R² 0.171, n 34943 | |  |

**h = 5일**

| 설명변수 | 미래 RS⁺ β | t | p | 미래 RS⁻ β | t | p | 미래 RV β | t | p |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 장중 RS⁺ 일 | 0.042*** | 6.33 | <0.001 | 0.045*** | 7.68 | <0.001 | 0.088*** | 7.26 | <0.001 |
| 장중 RS⁻ 일 | 0.144 | 1.52 | 0.129 | 0.052 | 1.85 | 0.065 | 0.196 | 1.60 | 0.109 |
| 장중 RS⁺ 주 | 0.025 | 0.58 | 0.562 | 0.058* | 2.33 | 0.020 | 0.083 | 1.30 | 0.194 |
| 장중 RS⁻ 주 | 0.092** | 3.05 | 0.002 | 0.043 | 1.65 | 0.099 | 0.135** | 2.94 | 0.003 |
| 장중 RS⁺ 월 | -0.001 | -0.01 | 0.992 | -0.007 | -0.07 | 0.941 | -0.007 | -0.04 | 0.967 |
| 장중 RS⁻ 월 | 0.232** | 2.75 | 0.006 | 0.223** | 2.70 | 0.007 | 0.454** | 2.81 | 0.005 |
| 적합도 | R² 0.117, n 34743 | |  | R² 0.096, n 34743 | |  | R² 0.133, n 34743 | |  |

**h = 22일**

| 설명변수 | 미래 RS⁺ β | t | p | 미래 RS⁻ β | t | p | 미래 RV β | t | p |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 장중 RS⁺ 일 | 0.014*** | 3.99 | <0.001 | 0.014*** | 5.58 | <0.001 | 0.028*** | 4.70 | <0.001 |
| 장중 RS⁻ 일 | 0.035 | 1.77 | 0.077 | 0.016* | 2.39 | 0.017 | 0.051 | 1.93 | 0.053 |
| 장중 RS⁺ 주 | 0.030 | 1.65 | 0.099 | 0.027 | 1.69 | 0.091 | 0.057 | 1.71 | 0.087 |
| 장중 RS⁻ 주 | -0.004 | -0.09 | 0.925 | 0.004 | 0.19 | 0.851 | 0.001 | 0.01 | 0.992 |
| 장중 RS⁺ 월 | -0.105 | -0.68 | 0.498 | -0.058 | -0.50 | 0.618 | -0.163 | -0.60 | 0.545 |
| 장중 RS⁻ 월 | 0.424 | 1.89 | 0.059 | 0.302* | 2.40 | 0.016 | 0.726* | 2.09 | 0.037 |
| 적합도 | R² 0.116, n 33893 | |  | R² 0.105, n 33893 | |  | R² 0.127, n 33893 | |  |

### A.4 60분봉 종목 패널 — 한국


**h = 1일**

| 설명변수 | 미래 RS⁺ β | t | p | 미래 RS⁻ β | t | p | 미래 RV β | t | p |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 장중 RS⁺ 일 | 0.025 | 1.21 | 0.225 | 0.038 | 1.51 | 0.132 | 0.063 | 1.73 | 0.084 |
| 장중 RS⁻ 일 | 0.069*** | 4.28 | <0.001 | 0.028 | 0.89 | 0.374 | 0.097** | 2.66 | 0.008 |
| 장중 RS⁺ 주 | 0.278*** | 4.24 | <0.001 | 0.115* | 2.52 | 0.012 | 0.394*** | 4.91 | <0.001 |
| 장중 RS⁻ 주 | 0.048 | 1.41 | 0.158 | 0.093* | 2.17 | 0.030 | 0.141** | 2.77 | 0.006 |
| 장중 RS⁺ 월 | 0.073 | 1.13 | 0.259 | 0.208** | 3.11 | 0.002 | 0.282* | 2.41 | 0.016 |
| 장중 RS⁻ 월 | 0.170* | 2.23 | 0.026 | 0.203* | 2.48 | 0.013 | 0.373** | 2.77 | 0.006 |
| 적합도 | R² 0.069, n 35265 | |  | R² 0.058, n 35265 | |  | R² 0.105, n 35265 | |  |

**h = 5일**

| 설명변수 | 미래 RS⁺ β | t | p | 미래 RS⁻ β | t | p | 미래 RV β | t | p |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 장중 RS⁺ 일 | 0.047*** | 3.61 | <0.001 | 0.027** | 2.82 | 0.005 | 0.074*** | 4.07 | <0.001 |
| 장중 RS⁻ 일 | 0.021* | 2.47 | 0.014 | 0.025* | 2.50 | 0.012 | 0.046** | 2.76 | 0.006 |
| 장중 RS⁺ 주 | 0.157** | 3.05 | 0.002 | 0.093** | 2.88 | 0.004 | 0.249*** | 3.96 | <0.001 |
| 장중 RS⁻ 주 | 0.071 | 1.45 | 0.148 | 0.026 | 0.53 | 0.597 | 0.097 | 1.22 | 0.223 |
| 장중 RS⁺ 월 | 0.104 | 1.63 | 0.103 | 0.199** | 2.99 | 0.003 | 0.303* | 2.49 | 0.013 |
| 장중 RS⁻ 월 | 0.175* | 2.35 | 0.019 | 0.264* | 2.45 | 0.014 | 0.438** | 2.70 | 0.007 |
| 적합도 | R² 0.141, n 35065 | |  | R² 0.159, n 35065 | |  | R² 0.209, n 35065 | |  |

**h = 22일**

| 설명변수 | 미래 RS⁺ β | t | p | 미래 RS⁻ β | t | p | 미래 RV β | t | p |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 장중 RS⁺ 일 | 0.017*** | 3.89 | <0.001 | 0.013*** | 3.67 | <0.001 | 0.030*** | 3.99 | <0.001 |
| 장중 RS⁻ 일 | 0.010* | 2.57 | 0.010 | 0.007* | 2.26 | 0.024 | 0.017** | 2.59 | 0.010 |
| 장중 RS⁺ 주 | 0.068*** | 4.08 | <0.001 | 0.052*** | 3.76 | <0.001 | 0.120*** | 4.83 | <0.001 |
| 장중 RS⁻ 주 | 0.050 | 1.63 | 0.103 | 0.024 | 1.54 | 0.124 | 0.074 | 1.68 | 0.092 |
| 장중 RS⁺ 월 | 0.118** | 2.59 | 0.010 | 0.176*** | 3.47 | <0.001 | 0.294** | 3.11 | 0.002 |
| 장중 RS⁻ 월 | 0.147* | 2.17 | 0.030 | 0.265** | 2.69 | 0.007 | 0.412* | 2.52 | 0.012 |
| 적합도 | R² 0.149, n 34215 | |  | R² 0.261, n 34215 | |  | R² 0.236, n 34215 | |  |

### A.5 60분봉 장중·밤사이 분리 — 미국 지수


**h = 1일**

| 설명변수 | 미래 RS⁺ β | t | p | 미래 RS⁻ β | t | p | 미래 RV β | t | p |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 장중 RS⁺ 일 | 0.064 | 0.90 | 0.370 | 0.154** | 3.21 | 0.001 | 0.218* | 2.22 | 0.026 |
| 장중 RS⁻ 일 | 0.939*** | 3.36 | <0.001 | -0.107 | -0.75 | 0.451 | 0.832*** | 3.85 | <0.001 |
| 장중 RS⁺ 주 | -0.189 | -1.56 | 0.119 | -0.057 | -0.74 | 0.462 | -0.246 | -1.40 | 0.160 |
| 장중 RS⁻ 주 | -0.194 | -0.82 | 0.415 | 0.150 | 1.13 | 0.257 | -0.044 | -0.15 | 0.885 |
| 장중 RS⁺ 월 | -0.338 | -0.69 | 0.491 | -0.047 | -0.29 | 0.773 | -0.384 | -0.71 | 0.478 |
| 장중 RS⁻ 월 | 0.108 | 0.23 | 0.819 | -0.063 | -0.29 | 0.769 | 0.044 | 0.08 | 0.933 |
| 밤사이 ON⁺ 일 | 0.464 | 1.73 | 0.083 | -0.064 | -1.20 | 0.231 | 0.400 | 1.73 | 0.083 |
| 밤사이 ON⁻ 일 | -0.145 | -1.23 | 0.218 | -0.077 | -1.00 | 0.319 | -0.222 | -1.21 | 0.225 |
| 밤사이 ON⁺ 주 | 0.121 | 0.98 | 0.328 | -0.323** | -2.67 | 0.008 | -0.203 | -1.20 | 0.231 |
| 밤사이 ON⁻ 주 | 0.591 | 1.62 | 0.105 | 0.436 | 1.56 | 0.118 | 1.026 | 1.85 | 0.065 |
| 밤사이 ON⁺ 월 | 0.073 | 0.43 | 0.664 | 0.042 | 0.26 | 0.797 | 0.115 | 0.65 | 0.513 |
| 밤사이 ON⁻ 월 | -0.079 | -0.46 | 0.644 | 0.279* | 2.22 | 0.026 | 0.201 | 0.89 | 0.375 |
| 적합도 | R² 0.597, n 698 | |  | R² 0.440, n 698 | |  | R² 0.648, n 698 | |  |

**h = 5일**

| 설명변수 | 미래 RS⁺ β | t | p | 미래 RS⁻ β | t | p | 미래 RV β | t | p |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 장중 RS⁺ 일 | -0.045 | -0.88 | 0.381 | 0.029 | 1.50 | 0.133 | -0.017 | -0.24 | 0.813 |
| 장중 RS⁻ 일 | 0.115 | 0.91 | 0.364 | 0.013 | 0.19 | 0.849 | 0.128 | 0.67 | 0.502 |
| 장중 RS⁺ 주 | -0.144 | -1.80 | 0.072 | 0.007 | 0.17 | 0.862 | -0.137 | -1.22 | 0.224 |
| 장중 RS⁻ 주 | 0.006 | 0.04 | 0.967 | 0.066 | 0.84 | 0.400 | 0.072 | 0.36 | 0.722 |
| 장중 RS⁺ 월 | -0.156 | -0.76 | 0.447 | -0.084 | -0.54 | 0.592 | -0.241 | -0.70 | 0.484 |
| 장중 RS⁻ 월 | 0.170 | 0.70 | 0.483 | 0.058 | 0.25 | 0.800 | 0.228 | 0.51 | 0.613 |
| 밤사이 ON⁺ 일 | 0.054 | 1.35 | 0.179 | -0.003 | -0.16 | 0.870 | 0.052 | 0.93 | 0.352 |
| 밤사이 ON⁻ 일 | 0.115* | 2.56 | 0.010 | 0.078*** | 3.88 | <0.001 | 0.193** | 2.99 | 0.003 |
| 밤사이 ON⁺ 주 | -0.128 | -1.06 | 0.291 | -0.147* | -2.11 | 0.035 | -0.275 | -1.48 | 0.140 |
| 밤사이 ON⁻ 주 | 0.606 | 1.86 | 0.063 | 0.161 | 1.36 | 0.174 | 0.767 | 1.73 | 0.084 |
| 밤사이 ON⁺ 월 | -0.016 | -0.15 | 0.882 | -0.081 | -0.59 | 0.552 | -0.097 | -0.45 | 0.649 |
| 밤사이 ON⁻ 월 | 0.132 | 1.01 | 0.312 | 0.216** | 2.89 | 0.004 | 0.348 | 1.83 | 0.068 |
| 적합도 | R² 0.567, n 694 | |  | R² 0.438, n 694 | |  | R² 0.549, n 694 | |  |

**h = 22일**

| 설명변수 | 미래 RS⁺ β | t | p | 미래 RS⁻ β | t | p | 미래 RV β | t | p |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 장중 RS⁺ 일 | 0.008 | 0.53 | 0.597 | 0.013* | 1.98 | 0.048 | 0.021 | 1.01 | 0.314 |
| 장중 RS⁻ 일 | 0.036 | 1.16 | 0.248 | 0.008 | 0.53 | 0.593 | 0.044 | 0.97 | 0.334 |
| 장중 RS⁺ 주 | 0.080 | 1.05 | 0.295 | 0.055 | 1.47 | 0.141 | 0.136 | 1.19 | 0.232 |
| 장중 RS⁻ 주 | -0.318 | -1.19 | 0.233 | -0.150 | -1.11 | 0.267 | -0.468 | -1.17 | 0.242 |
| 장중 RS⁺ 월 | -0.712 | -1.56 | 0.118 | -0.313 | -1.23 | 0.217 | -1.025 | -1.45 | 0.147 |
| 장중 RS⁻ 월 | 2.098 | 1.53 | 0.126 | 0.991 | 1.34 | 0.180 | 3.089 | 1.47 | 0.142 |
| 밤사이 ON⁺ 일 | 0.014 | 0.90 | 0.367 | 0.001 | 0.14 | 0.891 | 0.016 | 0.66 | 0.508 |
| 밤사이 ON⁻ 일 | 0.025** | 2.72 | 0.006 | 0.015** | 2.87 | 0.004 | 0.040** | 2.87 | 0.004 |
| 밤사이 ON⁺ 주 | -0.059 | -1.00 | 0.318 | -0.067 | -1.49 | 0.136 | -0.126 | -1.25 | 0.211 |
| 밤사이 ON⁻ 주 | 0.137 | 1.44 | 0.150 | 0.061 | 1.57 | 0.116 | 0.198 | 1.48 | 0.138 |
| 밤사이 ON⁺ 월 | -0.693 | -1.51 | 0.130 | -0.345 | -1.37 | 0.170 | -1.038 | -1.47 | 0.142 |
| 밤사이 ON⁻ 월 | 0.089 | 0.67 | 0.505 | 0.081 | 1.00 | 0.315 | 0.170 | 0.82 | 0.414 |
| 적합도 | R² 0.381, n 677 | |  | R² 0.307, n 677 | |  | R² 0.361, n 677 | |  |

### A.6 60분봉 장중·밤사이 분리 — 한국 지수


**h = 1일**

| 설명변수 | 미래 RS⁺ β | t | p | 미래 RS⁻ β | t | p | 미래 RV β | t | p |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 장중 RS⁺ 일 | 0.012 | 0.16 | 0.869 | -0.291*** | -3.53 | <0.001 | -0.279* | -2.09 | 0.036 |
| 장중 RS⁻ 일 | 0.196*** | 3.49 | <0.001 | 0.133 | 1.32 | 0.187 | 0.329* | 2.45 | 0.014 |
| 장중 RS⁺ 주 | 0.173 | 0.80 | 0.423 | 0.305* | 1.99 | 0.046 | 0.478* | 1.96 | 0.050 |
| 장중 RS⁻ 주 | 0.184 | 1.02 | 0.309 | 0.345 | 1.42 | 0.156 | 0.529** | 2.58 | 0.010 |
| 장중 RS⁺ 월 | -0.234 | -0.22 | 0.825 | 3.224* | 2.56 | 0.010 | 2.990* | 2.02 | 0.043 |
| 장중 RS⁻ 월 | 0.227 | 0.28 | 0.780 | -3.103* | -2.39 | 0.017 | -2.877* | -2.13 | 0.033 |
| 밤사이 ON⁺ 일 | -0.021* | -1.96 | 0.050 | -0.006 | -0.42 | 0.671 | -0.027 | -1.61 | 0.108 |
| 밤사이 ON⁻ 일 | 0.078 | 1.54 | 0.124 | 0.120 | 1.45 | 0.147 | 0.198* | 2.24 | 0.025 |
| 밤사이 ON⁺ 주 | -0.012 | -0.33 | 0.742 | -0.077 | -1.56 | 0.119 | -0.089 | -1.22 | 0.222 |
| 밤사이 ON⁻ 주 | 0.003 | 0.03 | 0.973 | -0.022 | -0.39 | 0.693 | -0.019 | -0.24 | 0.809 |
| 밤사이 ON⁺ 월 | 0.004 | 0.04 | 0.970 | 0.237 | 0.82 | 0.415 | 0.241 | 0.80 | 0.425 |
| 밤사이 ON⁻ 월 | 0.028 | 0.15 | 0.885 | 0.486 | 1.76 | 0.079 | 0.514 | 1.25 | 0.210 |
| 적합도 | R² 0.377, n 705 | |  | R² 0.317, n 705 | |  | R² 0.456, n 705 | |  |

**h = 5일**

| 설명변수 | 미래 RS⁺ β | t | p | 미래 RS⁻ β | t | p | 미래 RV β | t | p |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 장중 RS⁺ 일 | 0.044 | 0.71 | 0.477 | -0.029 | -0.42 | 0.671 | 0.015 | 0.12 | 0.906 |
| 장중 RS⁻ 일 | 0.048 | 0.92 | 0.356 | 0.096** | 3.25 | 0.001 | 0.144* | 2.32 | 0.020 |
| 장중 RS⁺ 주 | 0.036 | 0.31 | 0.758 | 0.147 | 1.43 | 0.152 | 0.184 | 0.96 | 0.337 |
| 장중 RS⁻ 주 | 0.235 | 1.29 | 0.196 | -0.029 | -0.19 | 0.851 | 0.206 | 0.69 | 0.490 |
| 장중 RS⁺ 월 | 0.218 | 0.26 | 0.793 | 2.854** | 2.77 | 0.006 | 3.072* | 2.06 | 0.040 |
| 장중 RS⁻ 월 | 0.291 | 0.53 | 0.600 | -1.995* | -2.18 | 0.029 | -1.703 | -1.56 | 0.119 |
| 밤사이 ON⁺ 일 | 0.002 | 0.38 | 0.703 | -0.004 | -0.43 | 0.669 | -0.002 | -0.14 | 0.886 |
| 밤사이 ON⁻ 일 | 0.052* | 2.44 | 0.015 | 0.038*** | 3.61 | <0.001 | 0.091** | 3.27 | 0.001 |
| 밤사이 ON⁺ 주 | -0.047 | -1.09 | 0.275 | 0.006 | 0.14 | 0.885 | -0.041 | -0.58 | 0.559 |
| 밤사이 ON⁻ 주 | -0.113 | -1.83 | 0.067 | -0.069 | -1.24 | 0.213 | -0.182 | -1.59 | 0.112 |
| 밤사이 ON⁺ 월 | -0.026 | -0.27 | 0.787 | 0.096 | 0.38 | 0.705 | 0.070 | 0.21 | 0.835 |
| 밤사이 ON⁻ 월 | 0.030 | 0.14 | 0.886 | 0.300 | 1.18 | 0.238 | 0.330 | 0.77 | 0.442 |
| 적합도 | R² 0.486, n 701 | |  | R² 0.506, n 701 | |  | R² 0.531, n 701 | |  |

**h = 22일**

| 설명변수 | 미래 RS⁺ β | t | p | 미래 RS⁻ β | t | p | 미래 RV β | t | p |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 장중 RS⁺ 일 | 0.022 | 0.78 | 0.435 | 0.028 | 0.64 | 0.525 | 0.050 | 0.70 | 0.487 |
| 장중 RS⁻ 일 | 0.033** | 3.04 | 0.002 | 0.023 | 1.29 | 0.196 | 0.056* | 1.99 | 0.047 |
| 장중 RS⁺ 주 | -0.016 | -0.17 | 0.867 | 0.052 | 0.43 | 0.665 | 0.036 | 0.17 | 0.867 |
| 장중 RS⁻ 주 | 0.107 | 1.23 | 0.217 | 0.059 | 0.60 | 0.547 | 0.166 | 0.91 | 0.365 |
| 장중 RS⁺ 월 | 1.161* | 2.55 | 0.011 | 1.512** | 2.70 | 0.007 | 2.673** | 2.68 | 0.007 |
| 장중 RS⁻ 월 | -0.480 | -1.16 | 0.247 | -0.479 | -0.94 | 0.349 | -0.959 | -1.06 | 0.289 |
| 밤사이 ON⁺ 일 | -0.004 | -0.94 | 0.350 | -0.005 | -0.97 | 0.330 | -0.009 | -0.96 | 0.336 |
| 밤사이 ON⁻ 일 | 0.015* | 1.96 | 0.050 | 0.016 | 1.59 | 0.112 | 0.032 | 1.81 | 0.070 |
| 밤사이 ON⁺ 주 | -0.028 | -1.58 | 0.114 | -0.035 | -1.28 | 0.200 | -0.063 | -1.42 | 0.155 |
| 밤사이 ON⁻ 주 | 0.019 | 0.51 | 0.609 | 0.051 | 1.50 | 0.134 | 0.071 | 0.99 | 0.322 |
| 밤사이 ON⁺ 월 | 0.216 | 1.28 | 0.201 | 0.250 | 1.25 | 0.210 | 0.467 | 1.27 | 0.203 |
| 밤사이 ON⁻ 월 | -0.238** | -2.71 | 0.007 | -0.341** | -3.16 | 0.002 | -0.580** | -3.00 | 0.003 |
| 적합도 | R² 0.581, n 684 | |  | R² 0.599, n 684 | |  | R² 0.597, n 684 | |  |

### A.7 60분봉 장중·밤사이 분리 — 미국 종목 패널


**h = 1일**

| 설명변수 | 미래 RS⁺ β | t | p | 미래 RS⁻ β | t | p | 미래 RV β | t | p |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 장중 RS⁺ 일 | 0.083*** | 5.27 | <0.001 | 0.175*** | 4.92 | <0.001 | 0.258*** | 5.51 | <0.001 |
| 장중 RS⁻ 일 | 0.342 | 1.45 | 0.147 | 0.036* | 2.56 | 0.010 | 0.378 | 1.62 | 0.105 |
| 장중 RS⁺ 주 | -0.050 | -0.46 | 0.642 | 0.024 | 0.50 | 0.614 | -0.026 | -0.17 | 0.865 |
| 장중 RS⁻ 주 | 0.236 | 1.94 | 0.053 | 0.075* | 2.05 | 0.041 | 0.311* | 2.16 | 0.031 |
| 장중 RS⁺ 월 | -0.008 | -0.14 | 0.885 | -0.005 | -0.08 | 0.939 | -0.013 | -0.11 | 0.909 |
| 장중 RS⁻ 월 | 0.078 | 1.31 | 0.189 | 0.175** | 2.66 | 0.008 | 0.254*** | 3.60 | <0.001 |
| 밤사이 ON⁺ 일 | 0.013 | 1.35 | 0.176 | 0.002 | 1.68 | 0.092 | 0.014 | 1.64 | 0.102 |
| 밤사이 ON⁻ 일 | -0.004 | -0.43 | 0.665 | -0.001 | -0.82 | 0.413 | -0.006 | -0.53 | 0.596 |
| 밤사이 ON⁺ 주 | -0.002 | -0.40 | 0.689 | 0.000 | 0.05 | 0.960 | -0.002 | -0.28 | 0.783 |
| 밤사이 ON⁻ 주 | 0.035 | 1.47 | 0.142 | 0.013 | 1.71 | 0.087 | 0.048 | 1.60 | 0.110 |
| 밤사이 ON⁺ 월 | -0.010 | -0.74 | 0.462 | 0.005 | 0.59 | 0.555 | -0.005 | -0.28 | 0.778 |
| 밤사이 ON⁻ 월 | 0.014* | 2.18 | 0.030 | 0.002 | 0.44 | 0.657 | 0.016** | 2.62 | 0.009 |
| 적합도 | R² 0.142, n 34893 | |  | R² 0.092, n 34893 | |  | R² 0.181, n 34893 | |  |

**h = 5일**

| 설명변수 | 미래 RS⁺ β | t | p | 미래 RS⁻ β | t | p | 미래 RV β | t | p |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 장중 RS⁺ 일 | 0.035*** | 7.37 | <0.001 | 0.043*** | 7.62 | <0.001 | 0.078*** | 8.01 | <0.001 |
| 장중 RS⁻ 일 | 0.134 | 1.53 | 0.127 | 0.048 | 1.85 | 0.065 | 0.182 | 1.61 | 0.108 |
| 장중 RS⁺ 주 | 0.011 | 0.25 | 0.800 | 0.055* | 2.13 | 0.034 | 0.066 | 0.99 | 0.322 |
| 장중 RS⁻ 주 | 0.084** | 2.92 | 0.003 | 0.041 | 1.57 | 0.117 | 0.125** | 2.71 | 0.007 |
| 장중 RS⁺ 월 | -0.009 | -0.10 | 0.917 | -0.013 | -0.15 | 0.884 | -0.022 | -0.13 | 0.899 |
| 장중 RS⁻ 월 | 0.208* | 2.41 | 0.016 | 0.208* | 2.51 | 0.012 | 0.417* | 2.54 | 0.011 |
| 밤사이 ON⁺ 일 | 0.003 | 1.82 | 0.068 | 0.001** | 3.05 | 0.002 | 0.005* | 2.32 | 0.020 |
| 밤사이 ON⁻ 일 | 0.005 | 1.75 | 0.080 | 0.003 | 1.83 | 0.068 | 0.008 | 1.85 | 0.065 |
| 밤사이 ON⁺ 주 | -0.003 | -0.72 | 0.472 | -0.001 | -0.22 | 0.827 | -0.004 | -0.59 | 0.554 |
| 밤사이 ON⁻ 주 | 0.019 | 1.19 | 0.234 | 0.006 | 1.19 | 0.234 | 0.025 | 1.22 | 0.222 |
| 밤사이 ON⁺ 월 | -0.007 | -0.45 | 0.650 | 0.005 | 0.44 | 0.658 | -0.002 | -0.08 | 0.934 |
| 밤사이 ON⁻ 월 | 0.016* | 2.00 | 0.046 | 0.005 | 1.01 | 0.312 | 0.021 | 1.96 | 0.050 |
| 적합도 | R² 0.136, n 34693 | |  | R² 0.101, n 34693 | |  | R² 0.148, n 34693 | |  |

**h = 22일**

| 설명변수 | 미래 RS⁺ β | t | p | 미래 RS⁻ β | t | p | 미래 RV β | t | p |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 장중 RS⁺ 일 | 0.012*** | 3.79 | <0.001 | 0.013*** | 5.33 | <0.001 | 0.025*** | 4.53 | <0.001 |
| 장중 RS⁻ 일 | 0.032 | 1.79 | 0.073 | 0.014* | 2.41 | 0.016 | 0.046 | 1.96 | 0.050 |
| 장중 RS⁺ 주 | 0.025 | 1.37 | 0.172 | 0.027 | 1.71 | 0.088 | 0.052 | 1.56 | 0.119 |
| 장중 RS⁻ 주 | -0.006 | -0.14 | 0.886 | 0.005 | 0.21 | 0.835 | -0.001 | -0.02 | 0.986 |
| 장중 RS⁺ 월 | -0.117 | -0.78 | 0.438 | -0.071 | -0.64 | 0.524 | -0.187 | -0.72 | 0.471 |
| 장중 RS⁻ 월 | 0.411 | 1.75 | 0.080 | 0.283* | 2.16 | 0.030 | 0.694 | 1.91 | 0.056 |
| 밤사이 ON⁺ 일 | 0.001* | 2.12 | 0.034 | 0.000** | 2.75 | 0.006 | 0.001** | 2.67 | 0.008 |
| 밤사이 ON⁻ 일 | 0.001* | 2.38 | 0.017 | 0.001* | 2.29 | 0.022 | 0.002* | 2.38 | 0.017 |
| 밤사이 ON⁺ 주 | -0.001 | -0.86 | 0.388 | -0.002 | -1.52 | 0.128 | -0.004 | -1.36 | 0.174 |
| 밤사이 ON⁻ 주 | 0.006 | 1.41 | 0.158 | 0.002 | 1.13 | 0.257 | 0.007 | 1.39 | 0.163 |
| 밤사이 ON⁺ 월 | -0.005 | -0.26 | 0.796 | 0.013 | 1.02 | 0.309 | 0.009 | 0.30 | 0.764 |
| 밤사이 ON⁻ 월 | 0.014* | 2.03 | 0.043 | 0.008 | 1.38 | 0.167 | 0.023 | 1.82 | 0.069 |
| 적합도 | R² 0.127, n 33843 | |  | R² 0.110, n 33843 | |  | R² 0.135, n 33843 | |  |

### A.8 60분봉 장중·밤사이 분리 — 한국 종목 패널


**h = 1일**

| 설명변수 | 미래 RS⁺ β | t | p | 미래 RS⁻ β | t | p | 미래 RV β | t | p |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 장중 RS⁺ 일 | 0.023 | 1.19 | 0.232 | 0.031 | 1.41 | 0.159 | 0.055 | 1.76 | 0.078 |
| 장중 RS⁻ 일 | 0.069*** | 4.03 | <0.001 | 0.025 | 0.79 | 0.428 | 0.094* | 2.46 | 0.014 |
| 장중 RS⁺ 주 | 0.259*** | 4.23 | <0.001 | 0.098* | 1.97 | 0.048 | 0.357*** | 4.43 | <0.001 |
| 장중 RS⁻ 주 | 0.031 | 0.74 | 0.458 | 0.074 | 1.61 | 0.107 | 0.104 | 1.80 | 0.072 |
| 장중 RS⁺ 월 | 0.050 | 0.75 | 0.456 | 0.177* | 2.51 | 0.012 | 0.227 | 1.94 | 0.052 |
| 장중 RS⁻ 월 | 0.137 | 1.86 | 0.063 | 0.171* | 1.98 | 0.047 | 0.308* | 2.27 | 0.023 |
| 밤사이 ON⁺ 일 | 0.003 | 0.60 | 0.550 | 0.012 | 1.58 | 0.115 | 0.015 | 1.32 | 0.188 |
| 밤사이 ON⁻ 일 | 0.006 | 0.97 | 0.330 | 0.011 | 1.03 | 0.304 | 0.017 | 1.21 | 0.227 |
| 밤사이 ON⁺ 주 | 0.014 | 0.88 | 0.378 | 0.009 | 0.62 | 0.538 | 0.023 | 1.00 | 0.319 |
| 밤사이 ON⁻ 주 | 0.002 | 0.22 | 0.824 | 0.005 | 0.65 | 0.516 | 0.007 | 0.58 | 0.559 |
| 밤사이 ON⁺ 월 | 0.027 | 1.68 | 0.094 | 0.009 | 0.53 | 0.597 | 0.036 | 1.29 | 0.197 |
| 밤사이 ON⁻ 월 | -0.007 | -0.27 | 0.790 | 0.014 | 0.89 | 0.376 | 0.007 | 0.22 | 0.826 |
| 적합도 | R² 0.071, n 35215 | |  | R² 0.063, n 35215 | |  | R² 0.111, n 35215 | |  |

**h = 5일**

| 설명변수 | 미래 RS⁺ β | t | p | 미래 RS⁻ β | t | p | 미래 RV β | t | p |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 장중 RS⁺ 일 | 0.044*** | 3.66 | <0.001 | 0.025** | 2.79 | 0.005 | 0.069*** | 4.11 | <0.001 |
| 장중 RS⁻ 일 | 0.020* | 2.20 | 0.028 | 0.025* | 2.31 | 0.021 | 0.045* | 2.46 | 0.014 |
| 장중 RS⁺ 주 | 0.146** | 3.18 | 0.001 | 0.080* | 2.30 | 0.021 | 0.225*** | 3.74 | <0.001 |
| 장중 RS⁻ 주 | 0.059 | 1.11 | 0.269 | 0.011 | 0.23 | 0.822 | 0.070 | 0.83 | 0.404 |
| 장중 RS⁺ 월 | 0.083 | 1.34 | 0.182 | 0.171* | 2.45 | 0.014 | 0.254* | 2.07 | 0.039 |
| 장중 RS⁻ 월 | 0.140 | 1.91 | 0.056 | 0.234* | 2.11 | 0.035 | 0.374* | 2.33 | 0.020 |
| 밤사이 ON⁺ 일 | 0.002 | 0.90 | 0.366 | 0.004 | 1.33 | 0.183 | 0.006 | 1.22 | 0.223 |
| 밤사이 ON⁻ 일 | 0.007* | 2.46 | 0.014 | 0.005 | 1.58 | 0.115 | 0.012* | 2.26 | 0.024 |
| 밤사이 ON⁺ 주 | 0.012 | 0.97 | 0.334 | 0.012 | 1.33 | 0.182 | 0.025 | 1.31 | 0.190 |
| 밤사이 ON⁻ 주 | -0.013 | -1.28 | 0.201 | -0.000 | -0.02 | 0.988 | -0.013 | -0.90 | 0.370 |
| 밤사이 ON⁺ 월 | 0.034 | 1.71 | 0.088 | 0.011 | 0.60 | 0.548 | 0.045 | 1.31 | 0.192 |
| 밤사이 ON⁻ 월 | -0.007 | -0.27 | 0.789 | 0.014 | 0.78 | 0.435 | 0.006 | 0.16 | 0.871 |
| 적합도 | R² 0.150, n 35015 | |  | R² 0.166, n 35015 | |  | R² 0.219, n 35015 | |  |

**h = 22일**

| 설명변수 | 미래 RS⁺ β | t | p | 미래 RS⁻ β | t | p | 미래 RV β | t | p |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 장중 RS⁺ 일 | 0.015*** | 4.37 | <0.001 | 0.012*** | 3.83 | <0.001 | 0.028*** | 4.42 | <0.001 |
| 장중 RS⁻ 일 | 0.010* | 2.44 | 0.015 | 0.007* | 2.14 | 0.032 | 0.017* | 2.43 | 0.015 |
| 장중 RS⁺ 주 | 0.060*** | 4.15 | <0.001 | 0.047** | 2.88 | 0.004 | 0.107*** | 4.29 | <0.001 |
| 장중 RS⁻ 주 | 0.043 | 1.56 | 0.119 | 0.019 | 1.19 | 0.235 | 0.063 | 1.47 | 0.140 |
| 장중 RS⁺ 월 | 0.085 | 1.77 | 0.076 | 0.142** | 2.79 | 0.005 | 0.227* | 2.35 | 0.019 |
| 장중 RS⁻ 월 | 0.103 | 1.60 | 0.110 | 0.225* | 2.22 | 0.027 | 0.327* | 2.03 | 0.043 |
| 밤사이 ON⁺ 일 | 0.001 | 1.21 | 0.227 | 0.002 | 1.49 | 0.137 | 0.003 | 1.39 | 0.166 |
| 밤사이 ON⁻ 일 | 0.002* | 2.27 | 0.023 | 0.002 | 1.47 | 0.140 | 0.004* | 2.04 | 0.041 |
| 밤사이 ON⁺ 주 | 0.005 | 0.79 | 0.429 | 0.003 | 0.69 | 0.490 | 0.008 | 0.81 | 0.420 |
| 밤사이 ON⁻ 주 | -0.005 | -1.63 | 0.103 | -0.003 | -0.72 | 0.474 | -0.008 | -1.17 | 0.242 |
| 밤사이 ON⁺ 월 | 0.047 | 1.83 | 0.067 | 0.031 | 1.36 | 0.174 | 0.079 | 1.67 | 0.095 |
| 밤사이 ON⁻ 월 | -0.011 | -0.63 | 0.527 | 0.006 | 0.44 | 0.663 | -0.005 | -0.18 | 0.855 |
| 적합도 | R² 0.167, n 34165 | |  | R² 0.272, n 34165 | |  | R² 0.253, n 34165 | |  |

### A.9 5분봉 지수 — 미국


**h = 1일**

| 설명변수 | 미래 RS⁺ β | t | p | 미래 RS⁻ β | t | p | 미래 RV β | t | p |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 장중 RS⁺ 일 | 0.294*** | 3.52 | <0.001 | 0.585 | 1.73 | 0.083 | 0.879* | 2.19 | 0.029 |
| 장중 RS⁻ 일 | -0.005 | -0.07 | 0.948 | -0.326 | -1.54 | 0.123 | -0.331 | -1.24 | 0.215 |
| 장중 RS⁺ 주 | 1.297* | 2.05 | 0.041 | 2.355* | 2.15 | 0.032 | 3.651* | 2.20 | 0.027 |
| 장중 RS⁻ 주 | -0.787 | -1.89 | 0.059 | -1.446 | -1.82 | 0.068 | -2.233 | -1.95 | 0.051 |
| 적합도 | R² 0.291, n 55 | |  | R² 0.321, n 55 | |  | R² 0.331, n 55 | |  |

**h = 5일**

| 설명변수 | 미래 RS⁺ β | t | p | 미래 RS⁻ β | t | p | 미래 RV β | t | p |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 장중 RS⁺ 일 | 0.093 | 0.68 | 0.495 | 0.226 | 1.18 | 0.237 | 0.319 | 0.98 | 0.327 |
| 장중 RS⁻ 일 | 0.066 | 0.94 | 0.346 | 0.020 | 0.26 | 0.797 | 0.085 | 0.59 | 0.557 |
| 장중 RS⁺ 주 | 1.735* | 2.35 | 0.019 | 2.115** | 2.63 | 0.008 | 3.850* | 2.52 | 0.012 |
| 장중 RS⁻ 주 | -1.175* | -2.00 | 0.045 | -1.483* | -2.33 | 0.020 | -2.658* | -2.20 | 0.028 |
| 적합도 | R² 0.379, n 51 | |  | R² 0.392, n 51 | |  | R² 0.391, n 51 | |  |

### A.10 5분봉 지수 — 한국


**h = 1일**

| 설명변수 | 미래 RS⁺ β | t | p | 미래 RS⁻ β | t | p | 미래 RV β | t | p |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 장중 RS⁺ 일 | -0.956*** | -3.65 | <0.001 | -0.922*** | -3.29 | <0.001 | -1.879*** | -3.85 | <0.001 |
| 장중 RS⁻ 일 | 1.420*** | 6.31 | <0.001 | 1.270*** | 5.23 | <0.001 | 2.690*** | 6.86 | <0.001 |
| 장중 RS⁺ 주 | 0.270 | 0.63 | 0.532 | 0.275 | 0.56 | 0.576 | 0.545 | 0.63 | 0.529 |
| 장중 RS⁻ 주 | -0.135 | -0.29 | 0.770 | -0.003 | -0.01 | 0.995 | -0.138 | -0.15 | 0.883 |
| 적합도 | R² 0.603, n 54 | |  | R² 0.464, n 54 | |  | R² 0.550, n 54 | |  |

**h = 5일**

| 설명변수 | 미래 RS⁺ β | t | p | 미래 RS⁻ β | t | p | 미래 RV β | t | p |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 장중 RS⁺ 일 | -0.590*** | -3.96 | <0.001 | -0.584** | -3.04 | 0.002 | -1.173*** | -3.46 | <0.001 |
| 장중 RS⁻ 일 | 0.724*** | 5.80 | <0.001 | 0.699*** | 4.32 | <0.001 | 1.423*** | 5.01 | <0.001 |
| 장중 RS⁺ 주 | 1.626*** | 4.46 | <0.001 | 1.692*** | 4.88 | <0.001 | 3.318*** | 4.71 | <0.001 |
| 장중 RS⁻ 주 | -1.189** | -2.84 | 0.005 | -1.166** | -2.79 | 0.005 | -2.355** | -2.83 | 0.005 |
| 적합도 | R² 0.409, n 50 | |  | R² 0.362, n 50 | |  | R² 0.386, n 50 | |  |

### A.11 10분봉 지수 — 미국


**h = 1일**

| 설명변수 | 미래 RS⁺ β | t | p | 미래 RS⁻ β | t | p | 미래 RV β | t | p |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 장중 RS⁺ 일 | 0.451*** | 3.29 | <0.001 | 0.723 | 1.67 | 0.094 | 1.174* | 2.16 | 0.031 |
| 장중 RS⁻ 일 | -0.025 | -0.38 | 0.702 | -0.328 | -1.64 | 0.101 | -0.353 | -1.40 | 0.160 |
| 장중 RS⁺ 주 | 0.556 | 1.63 | 0.102 | 0.935 | 1.95 | 0.052 | 1.492 | 1.96 | 0.050 |
| 장중 RS⁻ 주 | -0.231 | -0.93 | 0.351 | -0.408 | -1.04 | 0.297 | -0.639 | -1.13 | 0.259 |
| 적합도 | R² 0.357, n 55 | |  | R² 0.240, n 55 | |  | R² 0.311, n 55 | |  |

**h = 5일**

| 설명변수 | 미래 RS⁺ β | t | p | 미래 RS⁻ β | t | p | 미래 RV β | t | p |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 장중 RS⁺ 일 | 0.264 | 1.22 | 0.224 | 0.364 | 1.31 | 0.192 | 0.628 | 1.27 | 0.202 |
| 장중 RS⁻ 일 | 0.094 | 1.26 | 0.206 | 0.083 | 0.84 | 0.402 | 0.177 | 1.03 | 0.304 |
| 장중 RS⁺ 주 | 1.173 | 1.89 | 0.058 | 1.623* | 2.08 | 0.038 | 2.795* | 2.01 | 0.044 |
| 장중 RS⁻ 주 | -0.880* | -2.04 | 0.041 | -1.298* | -2.32 | 0.020 | -2.178* | -2.23 | 0.026 |
| 적합도 | R² 0.414, n 51 | |  | R² 0.392, n 51 | |  | R² 0.412, n 51 | |  |

### A.12 10분봉 지수 — 한국


**h = 1일**

| 설명변수 | 미래 RS⁺ β | t | p | 미래 RS⁻ β | t | p | 미래 RV β | t | p |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 장중 RS⁺ 일 | -0.583* | -2.06 | 0.039 | -0.845* | -2.29 | 0.022 | -1.428* | -2.29 | 0.022 |
| 장중 RS⁻ 일 | 0.881*** | 4.83 | <0.001 | 0.909*** | 3.54 | <0.001 | 1.790*** | 4.42 | <0.001 |
| 장중 RS⁺ 주 | -0.528 | -1.31 | 0.189 | 0.178 | 0.41 | 0.685 | -0.351 | -0.45 | 0.655 |
| 장중 RS⁻ 주 | 0.786* | 2.05 | 0.040 | 0.289 | 0.63 | 0.529 | 1.075 | 1.40 | 0.162 |
| 적합도 | R² 0.562, n 54 | |  | R² 0.317, n 54 | |  | R² 0.440, n 54 | |  |

**h = 5일**

| 설명변수 | 미래 RS⁺ β | t | p | 미래 RS⁻ β | t | p | 미래 RV β | t | p |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 장중 RS⁺ 일 | -0.260* | -1.97 | 0.049 | -0.285 | -1.88 | 0.061 | -0.546 | -1.93 | 0.054 |
| 장중 RS⁻ 일 | 0.381*** | 5.43 | <0.001 | 0.328*** | 3.85 | <0.001 | 0.709*** | 4.60 | <0.001 |
| 장중 RS⁺ 주 | 0.540 | 1.40 | 0.162 | 0.914 | 1.83 | 0.068 | 1.454 | 1.66 | 0.098 |
| 장중 RS⁻ 주 | -0.260 | -0.66 | 0.507 | -0.462 | -0.96 | 0.339 | -0.723 | -0.83 | 0.405 |
| 적합도 | R² 0.248, n 50 | |  | R² 0.230, n 50 | |  | R² 0.236, n 50 | |  |

### A.13 5분봉 종목 패널 — 미국


**h = 1일**

| 설명변수 | 미래 RS⁺ β | t | p | 미래 RS⁻ β | t | p | 미래 RV β | t | p |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 장중 RS⁺ 일 | 0.083** | 3.09 | 0.002 | 0.199*** | 4.77 | <0.001 | 0.282*** | 4.66 | <0.001 |
| 장중 RS⁻ 일 | 0.139*** | 4.86 | <0.001 | 0.086* | 2.45 | 0.014 | 0.225*** | 4.20 | <0.001 |
| 장중 RS⁺ 주 | 0.192*** | 4.68 | <0.001 | 0.072 | 1.17 | 0.241 | 0.265** | 2.93 | 0.003 |
| 장중 RS⁻ 주 | -0.020 | -0.57 | 0.566 | 0.078 | 1.21 | 0.226 | 0.058 | 0.66 | 0.509 |
| 적합도 | R² 0.075, n 2749 | |  | R² 0.070, n 2749 | |  | R² 0.097, n 2749 | |  |

**h = 5일**

| 설명변수 | 미래 RS⁺ β | t | p | 미래 RS⁻ β | t | p | 미래 RV β | t | p |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 장중 RS⁺ 일 | 0.052** | 2.97 | 0.003 | 0.066** | 2.85 | 0.004 | 0.118** | 3.03 | 0.002 |
| 장중 RS⁻ 일 | 0.045*** | 3.60 | <0.001 | 0.053*** | 3.50 | <0.001 | 0.098*** | 3.73 | <0.001 |
| 장중 RS⁺ 주 | 0.129* | 2.40 | 0.016 | 0.110* | 2.28 | 0.022 | 0.239* | 2.50 | 0.013 |
| 장중 RS⁻ 주 | -0.003 | -0.06 | 0.956 | 0.004 | 0.07 | 0.943 | 0.001 | 0.01 | 0.993 |
| 적합도 | R² 0.061, n 2549 | |  | R² 0.053, n 2549 | |  | R² 0.068, n 2549 | |  |

### A.14 5분봉 종목 패널 — 한국


**h = 1일**

| 설명변수 | 미래 RS⁺ β | t | p | 미래 RS⁻ β | t | p | 미래 RV β | t | p |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 장중 RS⁺ 일 | 0.011 | 0.17 | 0.864 | -0.020 | -0.28 | 0.779 | -0.009 | -0.07 | 0.944 |
| 장중 RS⁻ 일 | 0.427** | 2.90 | 0.004 | 0.359* | 2.56 | 0.010 | 0.786** | 2.76 | 0.006 |
| 장중 RS⁺ 주 | 0.152 | 1.22 | 0.221 | 0.315** | 2.83 | 0.005 | 0.467* | 2.05 | 0.040 |
| 장중 RS⁻ 주 | 0.081 | 1.06 | 0.290 | -0.027 | -0.27 | 0.788 | 0.055 | 0.33 | 0.740 |
| 적합도 | R² 0.243, n 2750 | |  | R² 0.246, n 2750 | |  | R² 0.285, n 2750 | |  |

**h = 5일**

| 설명변수 | 미래 RS⁺ β | t | p | 미래 RS⁻ β | t | p | 미래 RV β | t | p |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 장중 RS⁺ 일 | -0.035 | -0.90 | 0.371 | -0.013 | -0.35 | 0.725 | -0.048 | -0.63 | 0.526 |
| 장중 RS⁻ 일 | 0.169** | 2.89 | 0.004 | 0.128* | 2.32 | 0.020 | 0.297** | 2.63 | 0.009 |
| 장중 RS⁺ 주 | 0.385** | 3.24 | 0.001 | 0.390** | 3.00 | 0.003 | 0.775** | 3.17 | 0.002 |
| 장중 RS⁻ 주 | -0.031 | -0.34 | 0.736 | -0.023 | -0.21 | 0.832 | -0.053 | -0.27 | 0.786 |
| 적합도 | R² 0.235, n 2550 | |  | R² 0.251, n 2550 | |  | R² 0.257, n 2550 | |  |

### A.15 10분봉 종목 패널 — 미국


**h = 1일**

| 설명변수 | 미래 RS⁺ β | t | p | 미래 RS⁻ β | t | p | 미래 RV β | t | p |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 장중 RS⁺ 일 | 0.075* | 2.08 | 0.038 | 0.189*** | 3.56 | <0.001 | 0.264** | 3.19 | 0.001 |
| 장중 RS⁻ 일 | 0.097*** | 3.39 | <0.001 | 0.078* | 2.14 | 0.033 | 0.175** | 2.97 | 0.003 |
| 장중 RS⁺ 주 | 0.078 | 1.64 | 0.102 | 0.088 | 1.15 | 0.251 | 0.166 | 1.55 | 0.121 |
| 장중 RS⁻ 주 | 0.058 | 1.13 | 0.259 | 0.047 | 0.91 | 0.362 | 0.105 | 1.14 | 0.252 |
| 적합도 | R² 0.042, n 2749 | |  | R² 0.049, n 2749 | |  | R² 0.069, n 2749 | |  |

**h = 5일**

| 설명변수 | 미래 RS⁺ β | t | p | 미래 RS⁻ β | t | p | 미래 RV β | t | p |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 장중 RS⁺ 일 | 0.040* | 2.03 | 0.042 | 0.065** | 2.77 | 0.006 | 0.105* | 2.46 | 0.014 |
| 장중 RS⁻ 일 | 0.040*** | 4.25 | <0.001 | 0.034** | 2.77 | 0.006 | 0.073*** | 3.57 | <0.001 |
| 장중 RS⁺ 주 | 0.040 | 0.80 | 0.424 | 0.055 | 1.09 | 0.274 | 0.095 | 1.01 | 0.312 |
| 장중 RS⁻ 주 | 0.034 | 1.07 | 0.285 | 0.045 | 1.04 | 0.299 | 0.079 | 1.18 | 0.236 |
| 적합도 | R² 0.034, n 2549 | |  | R² 0.035, n 2549 | |  | R² 0.044, n 2549 | |  |

### A.16 10분봉 종목 패널 — 한국


**h = 1일**

| 설명변수 | 미래 RS⁺ β | t | p | 미래 RS⁻ β | t | p | 미래 RV β | t | p |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 장중 RS⁺ 일 | 0.030 | 0.62 | 0.533 | -0.037 | -0.39 | 0.697 | -0.006 | -0.05 | 0.960 |
| 장중 RS⁻ 일 | 0.325*** | 3.37 | <0.001 | 0.298* | 2.12 | 0.034 | 0.623** | 2.67 | 0.008 |
| 장중 RS⁺ 주 | 0.311* | 2.28 | 0.023 | 0.409*** | 5.55 | <0.001 | 0.720*** | 4.11 | <0.001 |
| 장중 RS⁻ 주 | -0.036 | -0.40 | 0.693 | -0.124 | -0.93 | 0.351 | -0.160 | -0.85 | 0.393 |
| 적합도 | R² 0.222, n 2750 | |  | R² 0.160, n 2750 | |  | R² 0.240, n 2750 | |  |

**h = 5일**

| 설명변수 | 미래 RS⁺ β | t | p | 미래 RS⁻ β | t | p | 미래 RV β | t | p |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 장중 RS⁺ 일 | -0.023 | -0.81 | 0.420 | -0.015 | -0.45 | 0.656 | -0.038 | -0.63 | 0.529 |
| 장중 RS⁻ 일 | 0.137*** | 3.88 | <0.001 | 0.098** | 2.78 | 0.005 | 0.235*** | 3.36 | <0.001 |
| 장중 RS⁺ 주 | 0.455*** | 4.36 | <0.001 | 0.505*** | 5.40 | <0.001 | 0.960*** | 5.04 | <0.001 |
| 장중 RS⁻ 주 | -0.146 | -1.55 | 0.121 | -0.166 | -1.38 | 0.168 | -0.312 | -1.49 | 0.137 |
| 적합도 | R² 0.227, n 2550 | |  | R² 0.214, n 2550 | |  | R² 0.243, n 2550 | |  |

## 부록 B. ETF와 일봉 대조군 총영향 검정

### B.1 60분봉 기간

**SPY (장중)**

| 목표 | h | 상승 총영향 Σβ⁺ | 하락 총영향 Σβ⁻ | 차이 Σβ⁻ − Σβ⁺ | β⁻ = β⁺ 동시 χ² (p) |
|---|---:|---:|---:|---:|---:|
| 미래 RS⁺ | 1 | -0.719 (t -1.05, p 0.294) | 2.519 (t 1.80, p 0.073) | 3.239 (t 1.55, p 0.121) | 9.8* (0.021) |
| 미래 RS⁺ | 5 | -0.261 (t -0.94, p 0.348) | 1.162* (t 2.06, p 0.040) | 1.423 (t 1.70, p 0.089) | 11.5** (0.009) |
| 미래 RS⁺ | 22 | -0.783 (t -1.26, p 0.209) | 1.756 (t 1.48, p 0.138) | 2.539 (t 1.41, p 0.160) | 11.4** (0.010) |
| 미래 RS⁻ | 1 | 0.268** (t 2.64, p 0.008) | 0.188 (t 1.06, p 0.290) | -0.080 (t -0.29, p 0.771) | 35.4*** (<0.001) |
| 미래 RS⁻ | 5 | 0.069 (t 0.53, p 0.597) | 0.338 (t 1.28, p 0.199) | 0.268 (t 0.69, p 0.491) | 6.5 (0.089) |
| 미래 RS⁻ | 22 | -0.304 (t -0.90, p 0.367) | 0.786 (t 1.23, p 0.220) | 1.089 (t 1.12, p 0.264) | 6.3 (0.100) |
| 미래 RV | 1 | -0.452 (t -0.60, p 0.546) | 2.707 (t 1.79, p 0.073) | 3.159 (t 1.40, p 0.161) | 7.2 (0.067) |
| 미래 RV | 5 | -0.192 (t -0.48, p 0.629) | 1.500 (t 1.86, p 0.063) | 1.692 (t 1.42, p 0.156) | 13.0** (0.005) |
| 미래 RV | 22 | -1.086 (t -1.13, p 0.256) | 2.542 (t 1.39, p 0.163) | 3.628 (t 1.31, p 0.192) | 10.1* (0.018) |

**KODEX 200 (장중)**

| 목표 | h | 상승 총영향 Σβ⁺ | 하락 총영향 Σβ⁻ | 차이 Σβ⁻ − Σβ⁺ | β⁻ = β⁺ 동시 χ² (p) |
|---|---:|---:|---:|---:|---:|
| 미래 RS⁺ | 1 | -0.049 (t -0.08, p 0.937) | 0.739 (t 1.41, p 0.159) | 0.788 (t 0.69, p 0.489) | 2.3 (0.521) |
| 미래 RS⁺ | 5 | 0.398 (t 0.50, p 0.616) | 0.330 (t 0.49, p 0.627) | -0.068 (t -0.05, p 0.963) | 9.6* (0.022) |
| 미래 RS⁺ | 22 | 0.909** (t 2.75, p 0.006) | -0.131 (t -0.40, p 0.689) | -1.039 (t -1.63, p 0.104) | 3.1 (0.375) |
| 미래 RS⁻ | 1 | 3.136** (t 3.15, p 0.002) | -1.547* (t -2.06, p 0.040) | -4.683** (t -2.69, p 0.007) | 10.1* (0.018) |
| 미래 RS⁻ | 5 | 2.692** (t 3.08, p 0.002) | -1.240 (t -1.83, p 0.068) | -3.933* (t -2.55, p 0.011) | 9.8* (0.020) |
| 미래 RS⁻ | 22 | 1.322** (t 2.77, p 0.006) | -0.264 (t -0.54, p 0.586) | -1.586 (t -1.69, p 0.092) | 10.3* (0.016) |
| 미래 RV | 1 | 3.088** (t 2.79, p 0.005) | -0.808 (t -0.93, p 0.351) | -3.896* (t -1.98, p 0.047) | 10.9* (0.012) |
| 미래 RV | 5 | 3.090* (t 2.13, p 0.033) | -0.910 (t -0.76, p 0.446) | -4.001 (t -1.52, p 0.129) | 5.6 (0.135) |
| 미래 RV | 22 | 2.231** (t 2.80, p 0.005) | -0.394 (t -0.49, p 0.623) | -2.625 (t -1.68, p 0.093) | 5.2 (0.155) |

**S&P 500 일봉 (같은 기간)**

| 목표 | h | 상승 총영향 Σβ⁺ | 하락 총영향 Σβ⁻ | 차이 Σβ⁻ − Σβ⁺ | β⁻ = β⁺ 동시 χ² (p) |
|---|---:|---:|---:|---:|---:|
| 미래 RS⁺ | 1 | -0.585 (t -1.27, p 0.205) | 1.780** (t 3.17, p 0.002) | 2.365* (t 2.33, p 0.020) | 38.8*** (<0.001) |
| 미래 RS⁺ | 5 | -0.358 (t -1.02, p 0.308) | 1.316** (t 3.28, p 0.001) | 1.674* (t 2.24, p 0.025) | 56.9*** (<0.001) |
| 미래 RS⁺ | 22 | -0.542 (t -1.35, p 0.178) | 0.976* (t 2.24, p 0.025) | 1.518 (t 1.81, p 0.070) | 270.4*** (<0.001) |
| 미래 RS⁻ | 1 | -0.216 (t -1.06, p 0.291) | 0.664* (t 2.54, p 0.011) | 0.880 (t 1.90, p 0.057) | 7.0 (0.073) |
| 미래 RS⁻ | 5 | -0.472 (t -1.40, p 0.163) | 0.778 (t 1.95, p 0.051) | 1.249 (t 1.70, p 0.088) | 23.8*** (<0.001) |
| 미래 RS⁻ | 22 | -0.564 (t -1.47, p 0.141) | 0.683 (t 1.66, p 0.097) | 1.247 (t 1.57, p 0.116) | 38.6*** (<0.001) |
| 미래 RV | 1 | -0.801 (t -1.28, p 0.201) | 2.444** (t 3.28, p 0.001) | 3.245* (t 2.38, p 0.017) | 30.9*** (<0.001) |
| 미래 RV | 5 | -0.830 (t -1.26, p 0.208) | 2.093** (t 2.74, p 0.006) | 2.924* (t 2.06, p 0.039) | 83.7*** (<0.001) |
| 미래 RV | 22 | -1.105 (t -1.41, p 0.158) | 1.659* (t 1.96, p 0.050) | 2.764 (t 1.70, p 0.089) | 136.7*** (<0.001) |

**KOSPI 일봉 (같은 기간)**

| 목표 | h | 상승 총영향 Σβ⁺ | 하락 총영향 Σβ⁻ | 차이 Σβ⁻ − Σβ⁺ | β⁻ = β⁺ 동시 χ² (p) |
|---|---:|---:|---:|---:|---:|
| 미래 RS⁺ | 1 | -0.371 (t -0.76, p 0.446) | 1.000* (t 2.04, p 0.041) | 1.371 (t 1.42, p 0.155) | 28.1*** (<0.001) |
| 미래 RS⁺ | 5 | -0.012 (t -0.03, p 0.979) | 0.641 (t 1.62, p 0.105) | 0.654 (t 0.78, p 0.437) | 24.2*** (<0.001) |
| 미래 RS⁺ | 22 | 0.242 (t 0.51, p 0.608) | 0.383 (t 1.23, p 0.219) | 0.142 (t 0.18, p 0.854) | 64.2*** (<0.001) |
| 미래 RS⁻ | 1 | 0.160 (t 0.35, p 0.728) | 0.591 (t 1.78, p 0.075) | 0.431 (t 0.56, p 0.572) | 6.1 (0.109) |
| 미래 RS⁻ | 5 | 0.097 (t 0.20, p 0.842) | 0.603 (t 1.62, p 0.105) | 0.505 (t 0.61, p 0.539) | 8.6* (0.034) |
| 미래 RS⁻ | 22 | 0.695 (t 1.58, p 0.114) | 0.075 (t 0.28, p 0.777) | -0.621 (t -0.94, p 0.346) | 33.0*** (<0.001) |
| 미래 RV | 1 | -0.211 (t -0.25, p 0.799) | 1.591* (t 2.37, p 0.018) | 1.802 (t 1.21, p 0.225) | 17.3*** (<0.001) |
| 미래 RV | 5 | 0.085 (t 0.09, p 0.925) | 1.244 (t 1.82, p 0.069) | 1.159 (t 0.75, p 0.456) | 20.3*** (<0.001) |
| 미래 RV | 22 | 0.937 (t 1.07, p 0.284) | 0.458 (t 0.81, p 0.419) | -0.479 (t -0.34, p 0.731) | 58.8*** (<0.001) |

### B.2 5분봉 기간 (60일)

**SPY 5분봉**

| 목표 | h | 상승 총영향 Σβ⁺ | 하락 총영향 Σβ⁻ | 차이 Σβ⁻ − Σβ⁺ | β⁻ = β⁺ 동시 χ² (p) |
|---|---:|---:|---:|---:|---:|
| 미래 RS⁺ | 1 | 1.549* (t 2.51, p 0.012) | -0.749 (t -1.79, p 0.073) | -2.298* (t -2.24, p 0.025) | 6.1* (0.048) |
| 미래 RS⁺ | 5 | 1.719* (t 2.43, p 0.015) | -1.012 (t -1.89, p 0.059) | -2.731* (t -2.20, p 0.028) | 6.6* (0.037) |
| 미래 RS⁻ | 1 | 3.054** (t 2.90, p 0.004) | -1.811* (t -2.37, p 0.018) | -4.865** (t -2.70, p 0.007) | 9.2** (0.010) |
| 미래 RS⁻ | 5 | 2.350** (t 2.91, p 0.004) | -1.428* (t -2.36, p 0.018) | -3.777** (t -2.68, p 0.007) | 8.9* (0.012) |
| 미래 RV | 1 | 4.603** (t 2.84, p 0.005) | -2.560* (t -2.24, p 0.025) | -7.163** (t -2.61, p 0.009) | 8.3* (0.016) |
| 미래 RV | 5 | 4.069** (t 2.71, p 0.007) | -2.440* (t -2.16, p 0.031) | -6.509* (t -2.48, p 0.013) | 8.1* (0.018) |

**KODEX 200 5분봉**

| 목표 | h | 상승 총영향 Σβ⁺ | 하락 총영향 Σβ⁻ | 차이 Σβ⁻ − Σβ⁺ | β⁻ = β⁺ 동시 χ² (p) |
|---|---:|---:|---:|---:|---:|
| 미래 RS⁺ | 1 | -0.068 (t -0.09, p 0.930) | 0.555 (t 0.77, p 0.443) | 0.622 (t 0.42, p 0.676) | 4.0 (0.136) |
| 미래 RS⁺ | 5 | 0.762 (t 0.84, p 0.401) | -0.239 (t -0.35, p 0.727) | -1.001 (t -0.63, p 0.528) | 4.4 (0.109) |
| 미래 RS⁻ | 1 | 0.356 (t 0.38, p 0.705) | 0.196 (t 0.24, p 0.813) | -0.160 (t -0.09, p 0.928) | 9.1* (0.011) |
| 미래 RS⁻ | 5 | 1.074 (t 0.96, p 0.338) | -0.389 (t -0.46, p 0.643) | -1.463 (t -0.75, p 0.454) | 2.3 (0.320) |
| 미래 RV | 1 | 0.288 (t 0.17, p 0.865) | 0.751 (t 0.49, p 0.626) | 0.463 (t 0.14, p 0.886) | 6.9* (0.032) |
| 미래 RV | 5 | 1.836 (t 0.91, p 0.365) | -0.627 (t -0.41, p 0.680) | -2.463 (t -0.70, p 0.486) | 3.1 (0.210) |

**S&P 500 일봉 (같은 60일)**

| 목표 | h | 상승 총영향 Σβ⁺ | 하락 총영향 Σβ⁻ | 차이 Σβ⁻ − Σβ⁺ | β⁻ = β⁺ 동시 χ² (p) |
|---|---:|---:|---:|---:|---:|
| 미래 RS⁺ | 1 | 0.196 (t 0.95, p 0.341) | 2.276* (t 2.42, p 0.015) | 2.080* (t 2.18, p 0.029) | 4.8 (0.093) |
| 미래 RS⁺ | 5 | -0.091 (t -0.65, p 0.513) | 1.722** (t 3.27, p 0.001) | 1.813** (t 2.79, p 0.005) | 8.4* (0.015) |
| 미래 RS⁻ | 1 | -0.269* (t -2.24, p 0.025) | 0.347 (t 1.16, p 0.248) | 0.615 (t 1.52, p 0.129) | 7.5* (0.024) |
| 미래 RS⁻ | 5 | -0.185** (t -2.67, p 0.008) | 0.138 (t 0.61, p 0.540) | 0.324 (t 1.12, p 0.263) | 1.5 (0.473) |
| 미래 RV | 1 | -0.073 (t -0.43, p 0.667) | 2.622** (t 2.77, p 0.006) | 2.695** (t 2.74, p 0.006) | 8.9* (0.011) |
| 미래 RV | 5 | -0.276 (t -1.52, p 0.128) | 1.861** (t 2.95, p 0.003) | 2.137** (t 2.66, p 0.008) | 12.6** (0.002) |

**KOSPI 일봉 (같은 60일)**

| 목표 | h | 상승 총영향 Σβ⁺ | 하락 총영향 Σβ⁻ | 차이 Σβ⁻ − Σβ⁺ | β⁻ = β⁺ 동시 χ² (p) |
|---|---:|---:|---:|---:|---:|
| 미래 RS⁺ | 1 | -0.708 (t -1.78, p 0.075) | 1.616* (t 2.15, p 0.031) | 2.324* (t 2.06, p 0.039) | 5.6 (0.060) |
| 미래 RS⁺ | 5 | -0.231* (t -2.01, p 0.044) | 0.711* (t 2.49, p 0.013) | 0.942* (t 2.38, p 0.017) | 6.3* (0.043) |
| 미래 RS⁻ | 1 | 0.025 (t 0.35, p 0.725) | 0.093 (t 0.55, p 0.584) | 0.068 (t 0.37, p 0.712) | 0.3 (0.859) |
| 미래 RS⁻ | 5 | -0.122* (t -2.40, p 0.016) | 0.272 (t 1.43, p 0.151) | 0.394* (t 2.08, p 0.038) | 4.3 (0.115) |
| 미래 RV | 1 | -0.684 (t -1.77, p 0.077) | 1.708** (t 2.59, p 0.010) | 2.392* (t 2.36, p 0.018) | 5.7 (0.059) |
| 미래 RV | 5 | -0.353*** (t -3.32, p <0.001) | 0.982*** (t 4.39, p <0.001) | 1.336*** (t 4.99, p <0.001) | 30.9*** (<0.001) |

## 부록 C. 재현 방법

```powershell
python scripts/step7b_intraday_semivar.py
```

- 산출물: `results/step7b/{60m,5m,10m}_{index,etf,daily,panel}_{US,KR}_{coef,equality}.csv`
- 장중·밤사이 분리: `60m_decomp_{index,panel}_{US,KR}_*.csv`
- 표본 요약: `sample_summary.csv`
